# HousingPulse AI — Actividad 1: Python, EDA y limpieza

**Máster:** Data Science e Inteligencia Artificial — Evolve 2026-2027  
**Proyecto:** HousingPulse AI  
**Autor:** Joan Poveda  
**Dataset:** Miami-Dade County, Florida (octubre de 2026)

## Objetivo

El objetivo de este notebook es explorar, analizar y preparar un conjunto de datos inmobiliarios del condado de Miami-Dade mediante Python y Pandas.

Se realizará un análisis exploratorio (EDA) para identificar problemas de calidad, como valores ausentes, duplicados, formatos incorrectos y valores atípicos.

Posteriormente, se aplicarán transformaciones justificadas para obtener una versión limpia de los datos (capa Silver), que podrá utilizarse en las siguientes fases del proyecto.

Este notebook constituye una propuesta individual de limpieza que posteriormente se comparará con las realizadas por el resto del equipo.

## 1. Importación de librerías

Se importa Pandas para la lectura, exploración y manipulación de los datos, y Path para gestionar las rutas de los archivos de forma independiente del sistema operativo.

In [ ]:
import pandas as pd
from pathlib import Path

print("Versión de Pandas:", pd.__version__)

## 2. Carga del dataset

Se utiliza el archivo original de datos inmobiliarios de Miami-Dade County correspondiente a la descarga de octubre de 2026.

Los datos se cargan sin aplicar modificaciones para conservar la información original y poder comparar posteriormente los resultados de la limpieza.

In [ ]:
# Localizar la carpeta de la actividad
BASE_DIR = Path.cwd()

if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

# Ruta del dataset original
ruta = BASE_DIR / "data" / "raw" / "realestatedataset-miami-dade-county-fl-2026-10.csv"

# Cargar el dataset
df = pd.read_csv(ruta, low_memory=False)

print("Número de filas:", df.shape[0])
print("Número de columnas:", df.shape[1])

display(df.head())

## 3. Inspección inicial de los datos

Antes de realizar cualquier transformación, se examinan las dimensiones, los nombres de las columnas, los tipos de datos y las estadísticas descriptivas.

Esta inspección permite comprender la estructura del dataset e identificar posibles problemas que deberán analizarse durante el EDA.

In [ ]:
print("COLUMNAS DEL DATASET")
print(df.columns.tolist())

print("\nINFORMACIÓN GENERAL")
df.info()

print("\nESTADÍSTICAS DESCRIPTIVAS")
display(df.describe())

### 3.1. Primeras observaciones

El dataset contiene 1.015.955 registros y 30 variables relacionadas con las características, ubicación y precios de propiedades inmobiliarias de Miami-Dade.

En la inspección inicial se han identificado:

- Valores ausentes en variables relevantes como superficie, año de construcción y precio de última venta.
- Posibles valores atípicos en habitaciones, baños, superficies y precios.
- Años de construcción que requieren revisión.
- Coordenadas geográficas que podrían no corresponder al área estudiada.

Estas observaciones no implican que todos los registros sean incorrectos. Será necesario realizar comprobaciones adicionales antes de aplicar transformaciones o eliminar datos.

## 4. Análisis de valores ausentes

Se analiza la cantidad y el porcentaje de valores nulos de cada variable para identificar qué información está incompleta.

Este análisis permitirá decidir qué columnas necesitan tratamiento y cuáles pueden conservar sus valores ausentes sin afectar al objetivo del proyecto.

No se eliminarán ni sustituirán valores hasta conocer su distribución y relevancia.

In [ ]:
# Número de valores ausentes por columna
nulos = df.isna().sum()

# Porcentaje de valores ausentes
porcentaje_nulos = (nulos / len(df)) * 100

# Tabla resumen
tabla_nulos = pd.DataFrame({
    "Valores ausentes": nulos,
    "Porcentaje (%)": porcentaje_nulos.round(2)
})

# Ordenar de mayor a menor porcentaje
tabla_nulos = tabla_nulos.sort_values(
    by="Porcentaje (%)",
    ascending=False
)

display(tabla_nulos)

## 5. Análisis de duplicados

Se comprueba si existen registros completamente duplicados y si el identificador `id` aparece repetido.

La presencia de identificadores repetidos no implica necesariamente que los registros sean duplicados. Podrían corresponder a diferentes anuncios o actualizaciones de una misma propiedad.

Por ello, se analizarán los resultados antes de decidir si procede eliminar algún registro.

In [ ]:
# Filas completamente duplicadas
duplicados_completos = df.duplicated().sum()

# Identificadores repetidos
duplicados_id = df.duplicated(subset=["id"]).sum()

# Identificadores únicos
ids_unicos = df["id"].nunique()

print("Filas completamente duplicadas:", duplicados_completos)
print("Filas con ID repetido:", duplicados_id)
print("Identificadores únicos:", ids_unicos)

### 5.1. Resultados del análisis de calidad

**Valores ausentes**

El análisis muestra diferencias importantes en la completitud de las variables:

- `basement`, `listing_type` y `fireplace` presentan más del 98 % de valores ausentes.
- `pool`, `waterfront` y `unit` presentan más del 89 %.
- `last_sold_price` y `last_sold_date` tienen aproximadamente un 35 % de valores ausentes.
- `sqft`, `beds` y `baths` presentan porcentajes relativamente bajos, inferiores al 6 %.
- `list_price` está prácticamente completo.

Por el momento, no se eliminarán columnas ni registros. Primero se analizará la utilidad de cada variable para HousingPulse AI.

**Duplicados**

Se han comprobado los registros completamente duplicados y los identificadores `id` repetidos.

Resultados:

- Registros totales: 1.015.955.
- Filas completamente duplicadas: 0.
- Identificadores repetidos: 0.
- Identificadores únicos: 1.015.955.

No se requiere eliminar duplicados por estos criterios. Queda pendiente comprobar si existen diferentes registros asociados a una misma propiedad.

## 6. Análisis de tipos de inmuebles

Se analiza la distribución de la variable `property_type` para conocer los tipos de propiedades presentes en el dataset.

Este análisis permitirá identificar qué categorías tienen suficientes registros y valorar si conviene limitar inicialmente HousingPulse AI a un único tipo de inmueble, tal como recomendó el tutor.

También se comprobará si existen categorías inconsistentes o poco representadas.

In [ ]:
# Número de propiedades por tipo
tipos_vivienda = df["property_type"].value_counts(dropna=False)

# Porcentaje de cada tipo
porcentajes = (
    df["property_type"]
    .value_counts(normalize=True, dropna=False)
    * 100
).round(2)

# Tabla resumen
tabla_tipos = pd.DataFrame({
    "Número de propiedades": tipos_vivienda,
    "Porcentaje (%)": porcentajes
})

display(tabla_tipos)

### 6.1. Visualización de tipos de inmuebles

Se representa gráficamente la cantidad de propiedades por categoría para facilitar la comparación entre los diferentes tipos de inmuebles.

El gráfico permitirá identificar las categorías predominantes y detectar posibles desequilibrios en la muestra.

In [ ]:
import matplotlib.pyplot as plt

# Crear gráfico de barras
tipos_vivienda.plot(
    kind="bar",
    figsize=(10, 5)
)

plt.title("Distribución de propiedades por tipo")
plt.xlabel("Tipo de propiedad")
plt.ylabel("Número de propiedades")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### 6.2. Interpretación de la distribución

El análisis de la variable `property_type` muestra que las viviendas unifamiliares (`single_family`) son la categoría predominante, seguidas de los condominios (`condo`).

También existe una presencia significativa de viviendas adosadas (`townhouse`) y apartamentos (`apartment`), mientras que otras categorías tienen una representación considerablemente menor.

**Decisión provisional:** se propone centrar la primera versión de HousingPulse AI en viviendas unifamiliares (`single_family`), siguiendo la recomendación del tutor de limitar inicialmente el proyecto a un tipo de inmueble.

Esta decisión queda pendiente de comprobar la disponibilidad y calidad de las variables necesarias para la estimación de precios.

## 7. Análisis de precios inmobiliarios

Se analizan las variables `list_price` y `last_sold_price` para conocer la distribución de los precios inmobiliarios.

El objetivo es identificar valores ausentes, precios iguales o inferiores a cero y posibles valores extremos.

Se utilizarán estadísticas descriptivas y percentiles para diferenciar propiedades de precio elevado que podrían ser válidas de registros que requieren una revisión adicional.

En esta fase no se eliminarán valores extremos automáticamente.

In [ ]:
# Variables de precios
columnas_precios = ["list_price", "last_sold_price"]

# Estadísticas descriptivas
display(
    df[columnas_precios].describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

# Comprobación de precios inválidos
for columna in columnas_precios:
    print(f"\nVariable: {columna}")

    print("Valores ausentes:", df[columna].isna().sum())
    print("Precios iguales a 0:", (df[columna] == 0).sum())
    print("Precios negativos:", (df[columna] < 0).sum())

### 7.1. Visualización de la distribución de precios

Se representa la distribución de los precios anunciados mediante un histograma.

Para facilitar la interpretación, se limita la visualización al percentil 99 de los precios positivos. Esta limitación afecta únicamente al gráfico y no modifica ni elimina registros del dataset.

Los valores superiores a ese percentil se analizarán posteriormente como posibles valores atípicos.

In [ ]:
import matplotlib.pyplot as plt

# Seleccionar precios positivos
precios = df.loc[df["list_price"] > 0, "list_price"]

# Calcular percentil 99
limite = precios.quantile(0.99)

# Crear histograma
plt.figure(figsize=(10, 5))

plt.hist(
    precios[precios <= limite],
    bins=50,
    edgecolor="black"
)

plt.title("Distribución de precios anunciados")
plt.xlabel("Precio anunciado ($)")
plt.ylabel("Número de propiedades")

plt.tight_layout()
plt.show()

print(f"Percentil 99: ${limite:,.2f}")

### 7.2. Interpretación de la distribución de precios

El histograma de `list_price` muestra una distribución asimétrica hacia la derecha, con una mayor concentración de propiedades en los rangos de precios más bajos y una cola prolongada hacia valores elevados.

El percentil 99 de los precios positivos se sitúa en **6.030.475 dólares**, lo que indica la presencia de un pequeño porcentaje de propiedades con precios superiores a esta cantidad.

Estos valores extremos podrían corresponder a propiedades de lujo, aunque también podrían existir errores en los registros.

**Decisión provisional:** no eliminar automáticamente los precios superiores al percentil 99. Se revisarán junto con otras características de las propiedades para determinar si son plausibles.

Además, se mantiene la distinción entre `list_price` (precio anunciado) y `last_sold_price` (precio de última venta), ya que representan conceptos diferentes.

## 8. Análisis de características de las propiedades

Se analizan las variables `beds`, `baths`, `sqft` y `year_built`, que describen las características físicas de los inmuebles.

El objetivo es identificar valores ausentes, valores fuera de rango y posibles inconsistencias.

Se diferenciarán los valores claramente inválidos de aquellos poco habituales que podrían corresponder a propiedades reales.

En esta fase no se modificarán los datos.

In [ ]:
# Variables que vamos a analizar
variables = ["beds", "baths", "sqft", "year_built"]

# Estadísticas descriptivas
display(
    df[variables].describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

# Comprobaciones de valores sospechosos
controles = {
    "Dormitorios negativos": (df["beds"] < 0).sum(),
    "Dormitorios superiores a 20": (df["beds"] > 20).sum(),
    "Baños negativos": (df["baths"] < 0).sum(),
    "Baños superiores a 20": (df["baths"] > 20).sum(),
    "Superficie igual a cero": (df["sqft"] == 0).sum(),
    "Superficie negativa": (df["sqft"] < 0).sum(),
    "Superficie superior a 20000 sqft": (df["sqft"] > 20000).sum(),
    "Construcción anterior a 1800": (df["year_built"] < 1800).sum(),
    "Construcción posterior a 2026": (df["year_built"] > 2026).sum()
}

display(
    pd.DataFrame(
        controles.items(),
        columns=["Comprobación", "Registros detectados"]
    )
)

### 8.1. Distribución de superficies

Se analiza la distribución de la superficie construida (`sqft`) mediante un histograma.

Para facilitar la visualización, se representan únicamente las superficies positivas hasta el percentil 99.

Esta selección afecta exclusivamente al gráfico y no modifica el dataset original.

In [ ]:
# Seleccionar superficies positivas
superficies = df.loc[df["sqft"] > 0, "sqft"]

# Calcular percentil 99
limite_sqft = superficies.quantile(0.99)

plt.figure(figsize=(10, 5))

plt.hist(
    superficies[superficies <= limite_sqft],
    bins=50,
    edgecolor="black"
)

plt.title("Distribución de superficies de las propiedades")
plt.xlabel("Superficie (sqft)")
plt.ylabel("Número de propiedades")

plt.tight_layout()
plt.show()

print(f"Percentil 99: {limite_sqft:,.2f} sqft")

### 8.2. Interpretación de las características físicas

El análisis de las características de las propiedades muestra que:

- La superficie mediana es de 1.382 sqft (aproximadamente 128,4 m²).
- El 99 % de las propiedades con superficie registrada no supera los 5.634 sqft.
- Se han detectado 1.605 propiedades con superficie igual a cero.
- Existen 616 propiedades con superficies superiores a 20.000 sqft.
- Se han identificado 62 registros con más de 20 dormitorios y 87 con más de 20 baños.
- Hay 9 registros con año de construcción anterior a 1800 y 286 posteriores a 2026.

La distribución de superficies presenta una asimetría positiva, con una mayor concentración de propiedades entre aproximadamente 800 y 2.000 sqft.

**Decisiones provisionales:**

1. Considerar las superficies iguales a cero como valores no válidos para calcular precios por unidad de superficie.
2. Revisar los registros con cantidades extremas de dormitorios y baños.
3. Investigar los años de construcción fuera del rango esperado.
4. No eliminar automáticamente propiedades con superficies elevadas, ya que podrían corresponder a inmuebles reales.

Los controles se han realizado sobre el dataset completo, por lo que será necesario repetirlos para el tipo de inmueble seleccionado.

## 9. Análisis temporal de las ventas

Se analiza la variable `last_sold_date`, que representa la fecha de la última venta registrada de una propiedad.

El objetivo es conocer la cobertura temporal del dataset, identificar fechas inválidas y estudiar la disponibilidad de operaciones recientes.

También se comprobará la coherencia entre la fecha de venta, el precio de última venta y el año de construcción.

Este análisis es especialmente relevante porque el tutor ha señalado el riesgo de utilizar características actuales de una vivienda para estimar precios de operaciones históricas.

La existencia de fechas y precios válidos no garantiza que las características de la vivienda correspondan al momento de la venta.

In [ ]:
# Convertir las fechas sin modificar el DataFrame original
fechas_venta = pd.to_datetime(
    df["last_sold_date"],
    errors="coerce"
)

# Fecha de referencia: octubre de 2026
fecha_referencia = pd.Timestamp("2026-10-31")

print("Fecha mínima:", fechas_venta.min())
print("Fecha máxima:", fechas_venta.max())

print("\nFechas ausentes o no interpretables:",
      fechas_venta.isna().sum())

print("Fechas posteriores a octubre de 2026:",
      (fechas_venta > fecha_referencia).sum())

print("Ventas desde 2020:",
      (fechas_venta >= "2020-01-01").sum())

print("Ventas desde 2023:",
      (fechas_venta >= "2023-01-01").sum())

print("Ventas desde 2025:",
      (fechas_venta >= "2025-01-01").sum())

# Comprobar coherencia entre venta y construcción
venta_antes_construccion = (
    fechas_venta.notna()
    & df["year_built"].notna()
    & (fechas_venta.dt.year < df["year_built"])
)

print("\nVentas anteriores al año de construcción:",
      venta_antes_construccion.sum())

# Comprobar disponibilidad conjunta de fecha y precio
fecha_con_precio = (
    fechas_venta.notna()
    & df["last_sold_price"].gt(0)
)

print("Ventas con fecha y precio positivo:",
      fecha_con_precio.sum())

### 9.1. Distribución anual de las ventas

Se representa el número de registros con fecha de última venta por año.

El objetivo es identificar los periodos con mayor cobertura y valorar si existen suficientes operaciones recientes para el desarrollo de HousingPulse AI.

El gráfico representa la última venta registrada de cada propiedad, no necesariamente todas las transacciones realizadas en el mercado.

In [ ]:
# Contar registros por año de última venta
ventas_por_anio = (
    fechas_venta
    .dropna()
    .dt.year
    .value_counts()
    .sort_index()
)

# Mostrar los últimos 20 años
ventas_recientes = ventas_por_anio[
    ventas_por_anio.index >= 2007
]

display(
    ventas_recientes.rename_axis("Año")
    .reset_index(name="Número de registros")
)

plt.figure(figsize=(12, 5))

ventas_recientes.plot(
    kind="bar"
)

plt.title("Distribución de últimas ventas registradas por año")
plt.xlabel("Año de última venta")
plt.ylabel("Número de propiedades")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 9.2. Interpretación del análisis temporal

El análisis de `last_sold_date` muestra que las fechas registradas abarcan desde el 1 de enero de 1900 hasta el 18 de junio de 2026.

Se han obtenido los siguientes resultados:

- 351.851 registros sin fecha de venta válida.
- 204.037 propiedades con última venta desde 2020.
- 92.999 propiedades con última venta desde 2023.
- 36.980 propiedades con última venta desde 2025.
- 660.546 registros con fecha de venta y precio positivo.
- 9.470 registros cuya fecha de última venta es anterior al año de construcción.

La mayor concentración anual se observa en 2021, con 43.408 registros.

**Limitaciones detectadas:**

1. El dataset contiene la última venta registrada de cada propiedad, no todas las operaciones históricas.
2. Existen posibles inconsistencias entre las fechas de venta y construcción.
3. No se ha confirmado que las características actuales de las viviendas correspondan al momento de la última venta.
4. La cobertura de 2026 es parcial.

**Decisión provisional:** conservar los registros originales y analizar las inconsistencias temporales antes de establecer las reglas de limpieza.

### 9.3. Investigación de inconsistencias temporales

Se examinan los registros cuya última venta es anterior al año de construcción.

El objetivo es conocer la magnitud de las diferencias temporales y determinar si se trata de errores evidentes o de situaciones que podrían tener una explicación inmobiliaria.

Se mostrarán ejemplos y se calculará la diferencia en años entre la construcción y la última venta.

No se modificarán ni eliminarán registros durante esta comprobación.

In [ ]:
# Seleccionar propiedades con venta anterior a construcción
df_incoherentes = df.loc[venta_antes_construccion].copy()

# Convertir la fecha para analizarla
df_incoherentes["fecha_venta"] = pd.to_datetime(
    df_incoherentes["last_sold_date"],
    errors="coerce"
)

# Calcular diferencia entre construcción y venta
df_incoherentes["diferencia_anios"] = (
    df_incoherentes["year_built"]
    - df_incoherentes["fecha_venta"].dt.year
)

# Resumen de las diferencias
display(df_incoherentes["diferencia_anios"].describe())

# Mostrar ejemplos sin direcciones ni identificadores
display(
    df_incoherentes[
        [
            "property_type",
            "year_built",
            "last_sold_date",
            "diferencia_anios",
            "last_sold_price"
        ]
    ].head(10)
)

# Clasificar por magnitud de la diferencia
print("Diferencia de 1 a 5 años:",
      df_incoherentes["diferencia_anios"].between(1, 5).sum())

print("Diferencia de 6 a 20 años:",
      df_incoherentes["diferencia_anios"].between(6, 20).sum())

print("Diferencia superior a 20 años:",
      (df_incoherentes["diferencia_anios"] > 20).sum())

### 9.4. Interpretación de las inconsistencias temporales

Se han identificado 9.470 registros cuya fecha de última venta es anterior al año de construcción.

La distribución de las diferencias es la siguiente:

- 4.047 registros presentan diferencias de 1 a 5 años.
- 1.559 registros presentan diferencias de 6 a 20 años.
- 3.864 registros presentan diferencias superiores a 20 años.
- La diferencia mediana es de 9 años y la máxima de 128 años.

Estas inconsistencias podrían deberse a reconstrucciones, ventas anteriores a la edificación o errores en los datos.

**Decisión provisional:** no eliminar automáticamente estos registros. Se documentará el problema y se estudiará su impacto en la selección de datos para el futuro modelo.

El análisis refuerza la necesidad de verificar que las características utilizadas para estimar un precio estaban disponibles en el momento correspondiente.

## 10. Análisis de coherencia geográfica

Se comprueba la ubicación de las propiedades mediante las variables `lat`, `lng`, `state` y `county_fips`.

El objetivo es detectar posibles registros ubicados fuera del área de estudio, Miami-Dade County (Florida).

Se utilizará un rango geográfico aproximado como control exploratorio. Este rango no representa los límites administrativos exactos del condado.

También se comprobará la coherencia de los códigos de estado y condado.

En esta fase no se eliminarán registros.

In [ ]:
# Rango geográfico aproximado para inspección
lat_min, lat_max = 25.0, 26.1
lng_min, lng_max = -80.9, -80.0

# Detectar coordenadas fuera del rango
fuera_rango = (
    ~df["lat"].between(lat_min, lat_max)
    | ~df["lng"].between(lng_min, lng_max)
)

print("Registros fuera del rango aproximado:",
      fuera_rango.sum())

print("\nEstados registrados:")
display(df["state"].value_counts(dropna=False))

print("\nCódigos de condado:")
display(df["county_fips"].value_counts(dropna=False))

# Mostrar ejemplos de coordenadas sospechosas
display(
    df.loc[
        fuera_rango,
        ["city", "state", "zip", "lat", "lng", "county_fips"]
    ].head(10)
)

### 10.1. Visualización de las coordenadas

Se representa la distribución espacial de una muestra reproducible de propiedades para detectar agrupaciones y posibles ubicaciones anómalas.

El gráfico utiliza longitud y latitud, sin necesidad de mostrar direcciones exactas.

La visualización permite identificar puntos alejados del área principal, aunque no sustituye una comprobación mediante los límites administrativos oficiales.

In [ ]:
# Seleccionar una muestra reproducible
muestra_geo = df.sample(
    n=min(10000, len(df)),
    random_state=42
)

plt.figure(figsize=(9, 7))

plt.scatter(
    muestra_geo["lng"],
    muestra_geo["lat"],
    s=3,
    alpha=0.4
)

plt.title("Distribución geográfica de propiedades")
plt.xlabel("Longitud")
plt.ylabel("Latitud")

plt.tight_layout()
plt.show()

### 10.2. Interpretación del análisis geográfico

El análisis de las coordenadas muestra que la mayoría de las propiedades se concentra en el área geográfica esperada de Miami-Dade County.

Se han identificado los siguientes resultados:

- 222 registros fuera del rango geográfico aproximado utilizado.
- 1.015.948 registros con estado `FL`.
- 5 registros con estados distintos de Florida.
- 2 registros sin estado.
- Todos los registros presentan `county_fips = 12086`.

También se han observado posibles inconsistencias entre ciudad, código postal y coordenadas.

**Decisiones provisionales:**

1. No eliminar automáticamente los 222 registros fuera del rango aproximado.
2. Revisar los estados distintos de Florida y los valores ausentes.
3. Investigar las inconsistencias entre ciudad, código postal y coordenadas.
4. Utilizar límites administrativos oficiales si posteriormente se necesita una validación geográfica precisa.

El análisis confirma que existen anomalías geográficas, aunque representan una proporción pequeña del dataset.

## 11. Análisis específico de viviendas unifamiliares

Tras explorar el dataset completo, se realiza un análisis específico de las propiedades clasificadas como `single_family`.

Esta categoría se ha seleccionado provisionalmente por ser la más representada y por permitir trabajar con un tipo de inmueble más homogéneo.

Se analizará la disponibilidad de precios, superficies, habitaciones, baños y fechas de venta.

El objetivo es comprobar si existen suficientes registros para desarrollar una primera versión de HousingPulse AI centrada en viviendas unifamiliares.

Esta selección es provisional y no modifica el dataset original.

In [ ]:
# Seleccionar viviendas unifamiliares
df_single = df.loc[
    df["property_type"] == "single_family"
].copy()

print("Total de viviendas unifamiliares:", len(df_single))

print(
    "Porcentaje del dataset:",
    round(len(df_single) / len(df) * 100, 2),
    "%"
)

# Variables principales
variables_clave = [
    "list_price",
    "last_sold_price",
    "last_sold_date",
    "beds",
    "baths",
    "sqft",
    "year_built"
]

# Completitud de variables
tabla_single = pd.DataFrame({
    "Valores disponibles": df_single[variables_clave].notna().sum(),
    "Valores ausentes": df_single[variables_clave].isna().sum(),
    "Porcentaje ausente (%)": (
        df_single[variables_clave].isna().mean() * 100
    ).round(2)
})

display(tabla_single)

# Registros con información básica para estimar precio anunciado
datos_basicos = (
    df_single["list_price"].gt(0)
    & df_single["sqft"].gt(0)
    & df_single["beds"].notna()
    & df_single["baths"].notna()
)

print(
    "\nViviendas con precio anunciado y características básicas:",
    datos_basicos.sum()
)

### 11.1. Análisis de precios en viviendas unifamiliares

Se estudia la distribución de los precios anunciados específicamente para las viviendas unifamiliares.

Este análisis permitirá comprobar si los precios y sus valores extremos presentan un comportamiento diferente al observado en el dataset completo.

Se utilizarán estadísticas descriptivas y percentiles, sin modificar los datos originales.

In [ ]:
# Precios positivos de viviendas unifamiliares
precios_single = df_single.loc[
    df_single["list_price"] > 0,
    "list_price"
]

# Estadísticas
display(
    precios_single.describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

# Histograma hasta el percentil 99
limite_single = precios_single.quantile(0.99)

plt.figure(figsize=(10, 5))

plt.hist(
    precios_single[precios_single <= limite_single],
    bins=50,
    edgecolor="black"
)

plt.title("Distribución de precios — Viviendas unifamiliares")
plt.xlabel("Precio anunciado ($)")
plt.ylabel("Número de viviendas")

plt.tight_layout()
plt.show()

print(f"Percentil 99: ${limite_single:,.2f}")

### 11.2. Interpretación de las viviendas unifamiliares

El dataset contiene 422.006 propiedades clasificadas como `single_family`, que representan el 41,54 % del total.

De ellas, 403.505 disponen simultáneamente de precio anunciado positivo, superficie positiva y datos de dormitorios y baños.

Las variables principales presentan una completitud elevada, lo que permite considerar esta categoría como candidata para el desarrollo inicial de HousingPulse AI.

**Análisis de precios:**

- Precio mediano: 615.000 dólares.
- Precio medio: aproximadamente 960.608 dólares.
- Percentil 99: aproximadamente 7.056.848 dólares.
- Precio mínimo positivo: 100 dólares.
- Precio máximo: 237 millones de dólares.

La distribución presenta asimetría positiva y valores extremos tanto en precios bajos como elevados.

**Decisión:** seleccionar provisionalmente `single_family` como tipo de inmueble principal y continuar con el diagnóstico de los precios anómalos antes de aplicar reglas de limpieza.

# 12. Pipeline de limpieza

Una vez completado el análisis exploratorio, se inicia la preparación de los datos para generar una primera capa Silver.

El pipeline se centrará inicialmente en viviendas unifamiliares (`single_family`), siguiendo el alcance provisional de HousingPulse AI.

Se conservará el DataFrame original sin modificaciones y se trabajará sobre una copia independiente.

Las transformaciones se aplicarán de forma progresiva, justificando cada decisión y registrando su impacto sobre el número de registros y la calidad de los datos.

In [ ]:
# Crear copia de trabajo
df_clean = df_single.copy()

# Guardar dimensiones iniciales
filas_iniciales = len(df_clean)
columnas_iniciales = len(df_clean.columns)

print("Filas iniciales:", filas_iniciales)
print("Columnas iniciales:", columnas_iniciales)

# Registro de acciones de limpieza
registro_limpieza = []

## 12.1. Normalización de tipos de datos

Se revisan y normalizan los tipos de las variables principales.

Las fechas de venta se convierten al tipo `datetime`, las variables numéricas se convierten a formatos adecuados y los códigos postales se tratan como identificadores, no como cantidades numéricas.

Esta transformación permite realizar comparaciones, filtros y validaciones de forma consistente.

Los valores que no puedan convertirse correctamente se registrarán como ausentes para su posterior revisión.

In [ ]:
# Convertir fecha de última venta
df_clean["last_sold_date"] = pd.to_datetime(
    df_clean["last_sold_date"],
    errors="coerce"
)

# Variables numéricas
columnas_numericas = [
    "beds",
    "baths",
    "sqft",
    "lot_sqft",
    "year_built",
    "list_price",
    "last_sold_price",
    "hoa_fee"
]

for columna in columnas_numericas:
    df_clean[columna] = pd.to_numeric(
        df_clean[columna],
        errors="coerce"
    )

# Convertir código postal a texto
df_clean["zip"] = (
    df_clean["zip"]
    .astype("Int64")
    .astype("string")
)

# Comprobar los tipos después de la conversión
display(df_clean[
    columnas_numericas + ["last_sold_date", "zip"]
].dtypes)

## 12.2. Análisis de precios anunciados sospechosos

Durante el EDA se detectaron viviendas unifamiliares con precios anunciados extremadamente bajos, incluyendo registros de 100 dólares.

Se investigará cuántas propiedades presentan precios iguales a cero, negativos o inferiores a determinados umbrales exploratorios.

También se analizará su relación con el estado de la propiedad (`status`) para comprobar si estos valores podrían corresponder a anuncios especiales o errores.

No se eliminarán registros hasta justificar los criterios de limpieza.

In [ ]:
# Comprobar precios bajos o inválidos
controles_precios = {
    "Precios ausentes": df_clean["list_price"].isna().sum(),
    "Precios iguales a cero": (df_clean["list_price"] == 0).sum(),
    "Precios negativos": (df_clean["list_price"] < 0).sum(),
    "Precios entre 0 y 1000 $": (
        (df_clean["list_price"] > 0) &
        (df_clean["list_price"] <= 1000)
    ).sum(),
    "Precios entre 1000 y 10000 $": (
        (df_clean["list_price"] > 1000) &
        (df_clean["list_price"] <= 10000)
    ).sum(),
    "Precios entre 10000 y 50000 $": (
        (df_clean["list_price"] > 10000) &
        (df_clean["list_price"] <= 50000)
    ).sum()
}

display(
    pd.DataFrame(
        controles_precios.items(),
        columns=["Comprobación", "Número de registros"]
    )
)

### 12.2.1. Relación entre precios bajos y estado de la propiedad

Se analiza la distribución de la variable `status` para conocer los estados inmobiliarios presentes en el dataset.

Posteriormente, se estudian los registros con precios anunciados positivos inferiores o iguales a 10.000 dólares.

El objetivo es determinar si los precios sospechosos se concentran en determinados estados y valorar posibles explicaciones antes de aplicar transformaciones.

In [ ]:
# Estados presentes en las viviendas unifamiliares
print("ESTADOS DE LAS PROPIEDADES")
display(df_clean["status"].value_counts(dropna=False))

# Seleccionar viviendas con precios bajos
precios_bajos = df_clean.loc[
    df_clean["list_price"].between(
        0, 10000, inclusive="right"
    )
    & (df_clean["list_price"] > 0)
].copy()

print("\nViviendas con precios positivos hasta 10.000 $:")
print(len(precios_bajos))

# Distribución por estado
display(
    precios_bajos["status"].value_counts(dropna=False)
)

# Mostrar ejemplos sin direcciones ni identificadores
display(
    precios_bajos[
        [
            "property_type",
            "status",
            "list_price",
            "beds",
            "baths",
            "sqft",
            "year_built"
        ]
    ].head(15)
)

### 12.2.2. Interpretación de los precios bajos

El análisis de las viviendas unifamiliares ha detectado:

- 20 registros sin precio anunciado.
- 2.101 registros con precio igual a cero.
- 14.141 registros con precios positivos inferiores o iguales a 10.000 dólares.
- Ningún registro con precio negativo.

De las 14.141 viviendas con precios positivos de hasta 10.000 dólares, 13.714 presentan el estado `other`, aproximadamente el 97 %.

Los ejemplos muestran viviendas con varias habitaciones y superficies habituales, pero con precios anunciados de entre 2.000 y 4.500 dólares.

**Hipótesis:** algunos registros podrían corresponder a alquileres u otras modalidades de anuncio, en lugar de precios de compraventa.

**Decisión provisional:** no eliminar automáticamente estos registros. Primero se investigará la relación entre `status`, `listing_type` y `list_price`, así como el significado de estas variables en la fuente original.

### 12.2.3. Investigación del tipo de anuncio

Se analiza la variable `listing_type` y su relación con los precios anunciados.

El objetivo es comprobar si los registros con precios bajos podrían corresponder a alquileres u otras modalidades inmobiliarias.

También se comparan los precios según el estado de la propiedad (`status`).

Este análisis permitirá determinar si es necesario separar diferentes tipos de anuncios antes de generar la capa Silver.

In [ ]:
# Tipos de anuncio disponibles
print("TIPOS DE ANUNCIO")
display(
    df_clean["listing_type"].value_counts(dropna=False)
)

# Tipos de anuncio entre viviendas con precios bajos
print("\nTIPOS DE ANUNCIO CON PRECIO HASTA 10.000 $")
display(
    precios_bajos["listing_type"].value_counts(dropna=False)
)

# Comparar precios por estado
print("\nPRECIOS SEGÚN ESTADO DE LA PROPIEDAD")

resumen_status = (
    df_clean.groupby("status")["list_price"]
    .agg(["count", "median", "mean", "min", "max"])
    .round(2)
)

display(resumen_status)

# Relación entre estado y tipo de anuncio
print("\nRELACIÓN ENTRE STATUS Y LISTING_TYPE")

display(
    pd.crosstab(
        df_clean["status"],
        df_clean["listing_type"],
        dropna=False
    )
)

### 12.2.4. Interpretación de precios según el estado

El análisis muestra diferencias importantes en los precios anunciados según la variable `status`.

Las propiedades clasificadas como `active`, `off_market` y `sold` presentan precios medianos de 764.900, 622.500 y 579.000 dólares, respectivamente.

En cambio, las propiedades con estado `other` presentan una mediana de 3.505 dólares y las clasificadas como `pending`, de 22.450 dólares.

Además, 13.714 de los 14.141 registros con precios positivos inferiores o iguales a 10.000 dólares pertenecen a la categoría `other`.

Estos resultados sugieren que el dataset podría contener diferentes modalidades de anuncios, posiblemente incluyendo alquileres.

Sin embargo, la variable `listing_type` presenta numerosos valores ausentes y no permite confirmar esta hipótesis.

**Decisión provisional:** no utilizar directamente todos los valores de `list_price` como precios de compraventa. Se identificarán los registros sospechosos y se conservarán las categorías originales hasta verificar su significado.

## 12.3. Identificación de precios sospechosos

Se crea una variable de control para identificar precios anunciados que requieren revisión.

Se consideran sospechosos, de forma provisional:

- Los precios ausentes.
- Los precios iguales o inferiores a cero.
- Los precios positivos de hasta 10.000 dólares.

El umbral de 10.000 dólares se utiliza como criterio exploratorio debido a la concentración de precios bajos en la categoría `other`.

Esta clasificación no demuestra que los registros sean incorrectos ni que correspondan necesariamente a alquileres.

Los registros se conservarán en la capa de trabajo para mantener la trazabilidad de los datos originales.

In [ ]:
# Identificar precios que necesitan revisión
df_clean["precio_sospechoso"] = (
    df_clean["list_price"].isna()
    | (df_clean["list_price"] <= 10000)
)

# Contar los registros marcados
print("Precios sospechosos:",
      df_clean["precio_sospechoso"].sum())

print("Precios no marcados:",
      (~df_clean["precio_sospechoso"]).sum())

# Distribución por estado
display(
    pd.crosstab(
        df_clean["status"],
        df_clean["precio_sospechoso"]
    )
)

# Registrar la transformación
registro_limpieza.append({
    "accion": "Identificación de precios sospechosos",
    "detalle": "Nulos o precios <= 10.000 dólares",
    "filas_antes": len(df_clean),
    "filas_despues": len(df_clean)
})

### 12.3.1. Revisión de propiedades pendientes

Se investigan las propiedades con estado `pending`, debido a que presentan una mediana de precios anunciados considerablemente inferior a la de otras categorías.

Se analizarán sus precios y características para identificar posibles patrones o inconsistencias.

No se modificarán los registros durante esta comprobación.

In [ ]:
# Seleccionar propiedades pendientes
df_pending = df_clean.loc[
    df_clean["status"] == "pending"
].copy()

print("Total de propiedades pending:", len(df_pending))

# Estadísticas de precios
display(
    df_pending["list_price"].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

# Comparar precios bajos y altos
print("Precios hasta 10.000 $:",
      df_pending["list_price"].le(10000).sum())

print("Precios superiores a 100.000 $:",
      df_pending["list_price"].gt(100000).sum())

# Mostrar ejemplos
display(
    df_pending[
        [
            "status",
            "listing_type",
            "list_price",
            "beds",
            "baths",
            "sqft"
        ]
    ].sort_values("list_price").head(15)
)

### 12.3.2. Resultados de la identificación de precios sospechosos

Se han identificado 16.262 viviendas unifamiliares con precios anunciados ausentes o inferiores o iguales a 10.000 dólares, incluyendo los precios iguales a cero.

Esto representa aproximadamente el 3,85 % de las 422.006 viviendas unifamiliares.

La distribución por estado muestra que:

- `other`: 13.718 registros sospechosos de 14.703.
- `pending`: 164 registros sospechosos de 344.
- `off_market`: 2.033 registros sospechosos de 379.269.
- `active`: 179 registros sospechosos de 10.507.
- `sold`: 167 registros sospechosos de 17.107.
- `coming_soon`: 1 registro sospechoso de 76.

El análisis específico de `pending` muestra una distribución de precios heterogénea, con una mediana de 22.450 dólares y una media superior al millón de dólares.

**Conclusión:** los resultados sugieren una posible mezcla de modalidades de anuncios, aunque no permiten identificar los alquileres de manera concluyente.

**Transformación aplicada:** creación de la variable booleana `precio_sospechoso`.

No se han eliminado registros. La variable permitirá diferenciar los precios que requieren revisión y facilitará posteriores decisiones de selección para el modelo.

## 12.4. Limpieza de superficies

La variable `sqft` representa la superficie de la propiedad en pies cuadrados.

Durante el análisis exploratorio se detectaron superficies iguales a cero y valores extremadamente elevados.

Para las viviendas unifamiliares se realizarán controles específicos de valores ausentes, superficies no positivas y superficies excepcionalmente grandes.

Las superficies iguales o inferiores a cero se considerarán no válidas para calcular el precio por unidad de superficie.

Los valores elevados se marcarán para revisión, sin eliminarlos automáticamente, ya que podrían corresponder a viviendas reales de gran tamaño.

In [ ]:
# Comprobaciones de superficie en viviendas unifamiliares
controles_sqft = {
    "Superficie ausente": df_clean["sqft"].isna().sum(),
    "Superficie igual a cero": (df_clean["sqft"] == 0).sum(),
    "Superficie negativa": (df_clean["sqft"] < 0).sum(),
    "Superficie entre 0 y 300 sqft": (
        (df_clean["sqft"] > 0) &
        (df_clean["sqft"] < 300)
    ).sum(),
    "Superficie superior a 10000 sqft": (
        df_clean["sqft"] > 10000
    ).sum(),
    "Superficie superior a 20000 sqft": (
        df_clean["sqft"] > 20000
    ).sum()
}

display(
    pd.DataFrame(
        controles_sqft.items(),
        columns=["Comprobación", "Número de registros"]
    )
)

# Estadísticas de superficies positivas
display(
    df_clean.loc[
        df_clean["sqft"] > 0, "sqft"
    ].describe(
        percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
    )
)

### 12.4.1. Identificación de superficies inválidas y extremas

Se crearán dos indicadores de calidad:

- `superficie_invalida`: identifica superficies ausentes, iguales a cero o negativas.
- `superficie_extrema`: identifica superficies positivas inferiores a 300 sqft o superiores a 20.000 sqft.

Los límites de 300 y 20.000 sqft son criterios exploratorios y no representan reglas definitivas de exclusión.

No se modificarán las superficies extremas sin una justificación adicional.

In [ ]:
# Identificar superficies inválidas
df_clean["superficie_invalida"] = (
    df_clean["sqft"].isna()
    | (df_clean["sqft"] <= 0)
)

# Identificar superficies extremas
df_clean["superficie_extrema"] = (
    ((df_clean["sqft"] > 0) & (df_clean["sqft"] < 300))
    | (df_clean["sqft"] > 20000)
)

print(
    "Superficies inválidas:",
    df_clean["superficie_invalida"].sum()
)

print(
    "Superficies extremas:",
    df_clean["superficie_extrema"].sum()
)

print(
    "Registros conservados:",
    len(df_clean)
)

# Registrar transformación
registro_limpieza.append({
    "accion": "Identificación de superficies problemáticas",
    "detalle": "Marcado de superficies inválidas y extremas",
    "filas_antes": len(df_clean),
    "filas_despues": len(df_clean)
})

### 12.4.2. Resultados del análisis de superficies

El análisis de las 422.006 viviendas unifamiliares muestra:

- 11.437 registros con superficie ausente.
- 755 registros con superficie igual a cero.
- Ningún registro con superficie negativa.
- 672 registros con superficie positiva inferior a 300 sqft.
- 238 registros con superficie superior a 20.000 sqft.

En total, se han identificado 12.192 registros con superficies ausentes o no positivas y 910 registros con superficies extremas.

La superficie mediana positiva es de 1.742 sqft, mientras que la máxima alcanza los 722.777 sqft.

**Decisiones de limpieza:**

1. Convertir las superficies iguales a cero en valores ausentes, ya que no representan una superficie construida útil para el análisis.
2. Mantener los valores ausentes sin sustituirlos por medias o medianas.
3. Conservar provisionalmente las superficies extremas, identificadas mediante `superficie_extrema`.
4. No eliminar registros completos por problemas de superficie en esta fase.

Estas decisiones permiten conservar la información original relevante y preparar una capa Silver con indicadores de calidad.

### 12.4.3. Normalización de superficies iguales a cero

Se sustituyen por valores ausentes (`NaN`) las superficies iguales a cero.

Esta transformación se justifica porque una superficie construida igual a cero no es válida para calcular indicadores como el precio por pie cuadrado.

Los registros no se eliminan y se conserva la variable `superficie_invalida` para identificar los problemas detectados originalmente.

Las superficies extremas permanecen sin modificar.

In [ ]:
# Contar superficies iguales a cero antes de limpiar
ceros_antes = (df_clean["sqft"] == 0).sum()

# Sustituir superficies iguales a cero por NaN
df_clean.loc[
    df_clean["sqft"] == 0, "sqft"
] = float("nan")

# Validar la transformación
ceros_despues = (df_clean["sqft"] == 0).sum()

print("Superficies iguales a cero antes:", ceros_antes)
print("Superficies iguales a cero después:", ceros_despues)
print("Valores ausentes después:", df_clean["sqft"].isna().sum())
print("Registros conservados:", len(df_clean))

# Registrar la transformación
registro_limpieza.append({
    "accion": "Normalización de superficies iguales a cero",
    "detalle": "Sustitución de sqft = 0 por NaN",
    "filas_antes": len(df_clean),
    "filas_despues": len(df_clean)
})

## 12.5. Análisis de dormitorios, baños y año de construcción

Se revisan las variables `beds`, `baths` y `year_built` para identificar posibles inconsistencias en las características de las viviendas unifamiliares.

Se comprobarán valores ausentes, negativos, iguales a cero y valores extremos.

Para el año de construcción se utilizará 2026 como referencia provisional, pendiente de confirmar la fecha real de extracción de la fuente.

Los valores sospechosos no se eliminarán automáticamente. Primero se analizará su frecuencia y se diferenciarán los errores evidentes de los casos poco habituales que podrían ser válidos.

In [ ]:
# Controles de dormitorios
controles_beds = {
    "Dormitorios ausentes": df_clean["beds"].isna().sum(),
    "Dormitorios iguales a cero": (df_clean["beds"] == 0).sum(),
    "Dormitorios negativos": (df_clean["beds"] < 0).sum(),
    "Dormitorios superiores a 10": (df_clean["beds"] > 10).sum(),
    "Dormitorios superiores a 20": (df_clean["beds"] > 20).sum()
}

# Controles de baños
controles_baths = {
    "Baños ausentes": df_clean["baths"].isna().sum(),
    "Baños iguales a cero": (df_clean["baths"] == 0).sum(),
    "Baños negativos": (df_clean["baths"] < 0).sum(),
    "Baños superiores a 10": (df_clean["baths"] > 10).sum(),
    "Baños superiores a 20": (df_clean["baths"] > 20).sum()
}

# Controles de año de construcción
controles_year = {
    "Año ausente": df_clean["year_built"].isna().sum(),
    "Año anterior a 1800": (df_clean["year_built"] < 1800).sum(),
    "Año posterior a 2026": (df_clean["year_built"] > 2026).sum(),
    "Año igual a cero": (df_clean["year_built"] == 0).sum()
}

# Mostrar resultados
print("CONTROLES DE DORMITORIOS")
display(pd.DataFrame(
    controles_beds.items(),
    columns=["Comprobación", "Registros"]
))

print("\nCONTROLES DE BAÑOS")
display(pd.DataFrame(
    controles_baths.items(),
    columns=["Comprobación", "Registros"]
))

print("\nCONTROLES DE AÑO DE CONSTRUCCIÓN")
display(pd.DataFrame(
    controles_year.items(),
    columns=["Comprobación", "Registros"]
))

### 12.5.1. Estadísticas descriptivas de características físicas

Se calculan estadísticas descriptivas de dormitorios, baños y año de construcción para contextualizar los valores sospechosos.

Se utilizarán percentiles para comprobar si los valores extremos se encuentran muy alejados de los rangos habituales.

Los resultados servirán para definir las siguientes transformaciones del pipeline.

In [ ]:
# Estadísticas de dormitorios, baños y construcción
display(
    df_clean[
        ["beds", "baths", "year_built"]
    ].describe(
        percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
    )
)

# Mostrar ejemplos de valores extremos
extremos = df_clean.loc[
    (df_clean["beds"] > 20)
    | (df_clean["baths"] > 20)
    | (df_clean["year_built"] < 1800)
    | (df_clean["year_built"] > 2026)
]

display(
    extremos[
        [
            "beds",
            "baths",
            "sqft",
            "year_built",
            "list_price",
            "status"
        ]
    ].head(15)
)

### 12.5.2. Resultados del análisis de características físicas

El análisis de las viviendas unifamiliares muestra:

**Dormitorios (`beds`):**
- 9.686 valores ausentes.
- 1.177 registros con cero dormitorios.
- 93 registros con más de 10 dormitorios.
- 4 registros con más de 20 dormitorios.
- Mediana de 3 dormitorios y máximo de 25.

**Baños (`baths`):**
- 9.235 valores ausentes.
- 1.097 registros con cero baños.
- 252 registros con más de 10 baños.
- 10 registros con más de 20 baños.
- Mediana de 2 baños y máximo de 30.

**Año de construcción (`year_built`):**
- 19.943 valores ausentes.
- 4 registros anteriores a 1800.
- 42 registros posteriores a 2026.
- Mediana de 1970.

Los valores extremos de dormitorios y baños podrían corresponder a propiedades especiales o errores de registro.

Los años posteriores a 2026 podrían representar fechas previstas de construcción.

**Decisión provisional:** identificar las anomalías mediante indicadores de calidad y evitar eliminaciones automáticas hasta disponer de una justificación suficiente.

### 12.5.3. Creación de indicadores de calidad

Se crean indicadores booleanos para identificar posibles problemas en dormitorios, baños y año de construcción.

Los criterios exploratorios son:

- `dormitorios_sospechosos`: dormitorios ausentes, negativos o superiores a 20.
- `banos_sospechosos`: baños ausentes, negativos o superiores a 20.
- `construccion_sospechosa`: año ausente, anterior a 1800 o posterior a 2026.

Los valores iguales a cero en dormitorios y baños se analizarán por separado, ya que pueden representar características especiales, datos no informados o errores.

Los indicadores no implican que todos los registros señalados sean incorrectos.

No se eliminarán registros en esta fase.

In [ ]:
# Indicador de dormitorios sospechosos
df_clean["dormitorios_sospechosos"] = (
    df_clean["beds"].isna()
    | (df_clean["beds"] < 0)
    | (df_clean["beds"] > 20)
)

# Indicador de baños sospechosos
df_clean["banos_sospechosos"] = (
    df_clean["baths"].isna()
    | (df_clean["baths"] < 0)
    | (df_clean["baths"] > 20)
)

# Indicador de año de construcción sospechoso
df_clean["construccion_sospechosa"] = (
    df_clean["year_built"].isna()
    | (df_clean["year_built"] < 1800)
    | (df_clean["year_built"] > 2026)
)

# Mostrar resultados
print(
    "Dormitorios sospechosos:",
    df_clean["dormitorios_sospechosos"].sum()
)

print(
    "Baños sospechosos:",
    df_clean["banos_sospechosos"].sum()
)

print(
    "Años de construcción sospechosos:",
    df_clean["construccion_sospechosa"].sum()
)

print("Registros conservados:", len(df_clean))

# Registrar la transformación
registro_limpieza.append({
    "accion": "Identificación de características físicas sospechosas",
    "detalle": "Indicadores para dormitorios, baños y construcción",
    "filas_antes": len(df_clean),
    "filas_despues": len(df_clean)
})

### 12.5.4. Investigación de dormitorios y baños iguales a cero

Se analizan las viviendas que presentan cero dormitorios o cero baños.

El objetivo es comprobar si estos valores están relacionados con superficies, precios o estados concretos.

No se sustituirán automáticamente por valores ausentes, ya que podrían tener interpretaciones diferentes según las características de la propiedad.

In [ ]:
# Viviendas con cero dormitorios
df_beds_cero = df_clean.loc[
    df_clean["beds"] == 0
].copy()

# Viviendas con cero baños
df_baths_cero = df_clean.loc[
    df_clean["baths"] == 0
].copy()

print("VIVIENDAS CON CERO DORMITORIOS")
print("Total:", len(df_beds_cero))

display(
    df_beds_cero[
        ["beds", "baths", "sqft", "list_price", "status"]
    ].head(10)
)

print("\nESTADOS DE VIVIENDAS CON CERO DORMITORIOS")
display(
    df_beds_cero["status"].value_counts(dropna=False)
)

print("\nVIVIENDAS CON CERO BAÑOS")
print("Total:", len(df_baths_cero))

display(
    df_baths_cero[
        ["beds", "baths", "sqft", "list_price", "status"]
    ].head(10)
)

print("\nESTADOS DE VIVIENDAS CON CERO BAÑOS")
display(
    df_baths_cero["status"].value_counts(dropna=False)
)

# Comprobar coincidencia entre ambos problemas
print(
    "\nViviendas con cero dormitorios y cero baños:",
    ((df_clean["beds"] == 0) & (df_clean["baths"] == 0)).sum()
)

### 12.5.5. Interpretación de dormitorios y baños iguales a cero

El análisis ha identificado:

- 1.177 viviendas con cero dormitorios.
- 1.097 viviendas con cero baños.
- 378 viviendas con cero dormitorios y cero baños simultáneamente.

Se han encontrado propiedades con cero dormitorios y superficies reducidas, que podrían corresponder a estudios u otras configuraciones especiales.

Sin embargo, también aparecen viviendas de superficies habituales y precios elevados con cero dormitorios o baños, lo que sugiere posibles datos incompletos.

**Decisiones de limpieza:**

1. Conservar los dormitorios iguales a cero, ya que podrían representar una característica real.
2. Identificar estos registros mediante un indicador específico.
3. Convertir provisionalmente los baños iguales a cero en valores ausentes (`NaN`), interpretándolos como información no confirmada.
4. Conservar un indicador de los baños originalmente iguales a cero.
5. Mantener los valores extremos de dormitorios y baños para revisión posterior.

No se eliminarán registros completos.

### 12.5.6. Normalización de baños iguales a cero

Se sustituyen los valores iguales a cero de `baths` por valores ausentes.

Esta decisión se adopta como criterio provisional de calidad para viviendas unifamiliares, ya que no puede confirmarse que los ceros representen la ausencia real de baños.

Antes de realizar la transformación, se crean indicadores para conservar la información sobre los valores originales.

La columna `beds` no se modifica.

In [ ]:
# Guardar indicadores antes de modificar los valores
df_clean["dormitorios_cero"] = df_single["beds"] == 0
df_clean["banos_cero_original"] = df_single["baths"] == 0

# Contar los baños iguales a cero antes
banos_cero_antes = (df_clean["baths"] == 0).sum()

# Sustituir baños iguales a cero por NaN
df_clean.loc[
    df_clean["baths"] == 0, "baths"
] = float("nan")

# Validar la transformación
banos_cero_despues = (df_clean["baths"] == 0).sum()

print("Baños iguales a cero antes:", banos_cero_antes)
print("Baños iguales a cero después:", banos_cero_despues)
print("Valores ausentes actuales:", df_clean["baths"].isna().sum())

print(
    "Dormitorios iguales a cero conservados:",
    (df_clean["beds"] == 0).sum()
)

print("Registros conservados:", len(df_clean))

# Registrar transformación
registro_limpieza.append({
    "accion": "Normalización de baños iguales a cero",
    "detalle": "Sustitución de baths = 0 por NaN, conservando indicador",
    "filas_antes": len(df_clean),
    "filas_despues": len(df_clean)
})

### 12.5.7. Investigación de años de construcción extremos

Se analizan los registros con años de construcción anteriores a 1800 y posteriores a 2026.

Los años antiguos podrían corresponder a errores de registro o propiedades históricas.

Los años posteriores a 2026 podrían indicar construcciones previstas o promociones inmobiliarias todavía no finalizadas.

Se comprobarán sus características y su relación con el estado de la propiedad antes de decidir cualquier transformación.

In [ ]:
# Viviendas con año anterior a 1800
construcciones_antiguas = df_clean.loc[
    df_clean["year_built"] < 1800
].copy()

# Viviendas con año posterior a 2026
construcciones_futuras = df_clean.loc[
    df_clean["year_built"] > 2026
].copy()

print("CONSTRUCCIONES ANTERIORES A 1800")

display(
    construcciones_antiguas[
        [
            "year_built",
            "status",
            "listing_type",
            "beds",
            "baths",
            "sqft",
            "list_price"
        ]
    ]
)

print("\nCONSTRUCCIONES POSTERIORES A 2026")

display(
    construcciones_futuras[
        [
            "year_built",
            "status",
            "listing_type",
            "beds",
            "baths",
            "sqft",
            "list_price"
        ]
    ].head(15)
)

print("\nESTADOS DE CONSTRUCCIONES FUTURAS")

display(
    construcciones_futuras["status"].value_counts(dropna=False)
)

print("\nTIPOS DE ANUNCIO DE CONSTRUCCIONES FUTURAS")

display(
    construcciones_futuras["listing_type"].value_counts(dropna=False)
)

### 12.5.8. Resultados del análisis de años de construcción

Se han identificado cuatro viviendas unifamiliares con años de construcción anteriores a 1800: 1730, 1741, 1790 y 1798.

Estos valores requieren revisión, ya que podrían corresponder a errores de registro o propiedades históricas.

También se han identificado 42 viviendas con años de construcción posteriores a 2026.

De estas propiedades:

- 35 presentan el estado `active`.
- 17 tienen `listing_type = new_construction`.
- 1 presenta el estado `coming_soon`.

Estos resultados sugieren que algunas fechas futuras podrían corresponder a construcciones previstas.

**Decisión de limpieza:**

1. Conservar los años de construcción originales.
2. Identificar por separado las fechas anteriores a 1800.
3. Identificar las construcciones posteriores a 2026.
4. No sustituir ni eliminar estos valores sin información adicional.
5. Evitar interpretar los años futuros como fechas de construcción ya completada.

La referencia temporal de 2026 corresponde al periodo del dataset y deberá contrastarse con su fecha real de extracción.

### 12.5.9. Clasificación de anomalías temporales

Se crean dos indicadores adicionales para diferenciar los años de construcción antiguos y futuros.

- `construccion_muy_antigua`: año anterior a 1800.
- `construccion_futura`: año posterior a 2026.

Estas variables complementan el indicador general `construccion_sospechosa`.

Los valores originales de `year_built` se mantienen sin modificar.

In [ ]:
# Identificar construcciones anteriores a 1800
df_clean["construccion_muy_antigua"] = (
    df_clean["year_built"] < 1800
)

# Identificar construcciones posteriores a 2026
df_clean["construccion_futura"] = (
    df_clean["year_built"] > 2026
)

# Mostrar resultados
print(
    "Construcciones anteriores a 1800:",
    df_clean["construccion_muy_antigua"].sum()
)

print(
    "Construcciones posteriores a 2026:",
    df_clean["construccion_futura"].sum()
)

print(
    "Años de construcción ausentes:",
    df_clean["year_built"].isna().sum()
)

print("Registros conservados:", len(df_clean))

# Registrar transformación
registro_limpieza.append({
    "accion": "Clasificación de años de construcción sospechosos",
    "detalle": "Indicadores para años anteriores a 1800 y posteriores a 2026",
    "filas_antes": len(df_clean),
    "filas_despues": len(df_clean)
})

## 12.6. Análisis de calidad geográfica

Se revisan las variables geográficas de las viviendas unifamiliares:

- `state`: estado de Estados Unidos.
- `county_fips`: identificador administrativo del condado.
- `zip`: código postal.
- `lat`: latitud.
- `lng`: longitud.

El dataset está orientado al condado de Miami-Dade, Florida.

Se comprobarán valores ausentes, estados diferentes de Florida y coordenadas alejadas del área geográfica esperada.

Para la revisión inicial se utilizará una caja geográfica aproximada:

- Latitud entre 25.0 y 26.1.
- Longitud entre -80.9 y -80.0.

Esta caja se utiliza únicamente para detectar posibles anomalías y no representa el límite administrativo exacto de Miami-Dade.

No se eliminarán registros basándose únicamente en este criterio.

In [ ]:
# Revisar estados presentes
print("ESTADOS")
display(
    df_clean["state"].value_counts(dropna=False)
)

# Revisar condados
print("\nCOUNTY FIPS")
display(
    df_clean["county_fips"].value_counts(dropna=False)
)

# Comprobar coordenadas ausentes
print("\nCOORDENADAS AUSENTES")

print("Latitud ausente:", df_clean["lat"].isna().sum())
print("Longitud ausente:", df_clean["lng"].isna().sum())

# Identificar coordenadas fuera de la caja aproximada
fuera_zona = (
    ~df_clean["lat"].between(25.0, 26.1)
    | ~df_clean["lng"].between(-80.9, -80.0)
)

print(
    "\nRegistros fuera de la caja aproximada:",
    fuera_zona.sum()
)

# Examinar ejemplos
display(
    df_clean.loc[
        fuera_zona,
        [
            "city",
            "state",
            "zip",
            "lat",
            "lng",
            "county_fips",
            "list_price"
        ]
    ].head(15)
)

### 12.6.1. Resultados del análisis geográfico

El análisis geográfico de las 422.006 viviendas unifamiliares muestra:

- 422.001 registros con `state = FL`.
- 3 registros con `state = AL`.
- 1 registro con `state = TN`.
- 1 registro con estado ausente.
- Todos los registros presentan `county_fips = 12086`.
- No existen coordenadas ausentes.
- 52 registros se encuentran fuera de la caja geográfica aproximada definida para el análisis.

Se han detectado coordenadas claramente alejadas de Miami-Dade, incluyendo ubicaciones fuera del estado de Florida.

También existen registros cercanos a los límites de la zona estudiada, por lo que no todos los casos señalados pueden considerarse errores.

**Decisión de limpieza:**

1. Identificar las coordenadas fuera de la zona aproximada.
2. Identificar los estados diferentes de Florida o ausentes.
3. Conservar los registros originales y sus coordenadas.
4. No eliminar automáticamente los registros sospechosos.
5. Documentar que la validación geográfica definitiva requeriría utilizar los límites administrativos oficiales de Miami-Dade.

Los indicadores permitirán filtrar posteriormente los registros que requieran revisión.

### 12.6.2. Creación de indicadores geográficos

Se crean dos indicadores de calidad:

- `coordenadas_sospechosas`: identifica coordenadas ausentes o fuera de la caja geográfica aproximada.
- `estado_sospechoso`: identifica registros cuyo estado no sea Florida, incluyendo valores ausentes.

Los indicadores no modifican las coordenadas ni eliminan registros.

Su objetivo es facilitar la detección y revisión de posibles inconsistencias geográficas.

In [ ]:
# Identificar coordenadas fuera de la zona aproximada
df_clean["coordenadas_sospechosas"] = (
    df_clean["lat"].isna()
    | df_clean["lng"].isna()
    | ~df_clean["lat"].between(25.0, 26.1)
    | ~df_clean["lng"].between(-80.9, -80.0)
)

# Identificar estados distintos de Florida o ausentes
df_clean["estado_sospechoso"] = (
    df_clean["state"].isna()
    | (df_clean["state"] != "FL")
)

print(
    "Coordenadas sospechosas:",
    df_clean["coordenadas_sospechosas"].sum()
)

print(
    "Estados sospechosos:",
    df_clean["estado_sospechoso"].sum()
)

print(
    "Registros con alguna anomalía geográfica:",
    (
        df_clean["coordenadas_sospechosas"]
        | df_clean["estado_sospechoso"]
    ).sum()
)

print("Registros conservados:", len(df_clean))

# Registrar transformación
registro_limpieza.append({
    "accion": "Identificación de anomalías geográficas",
    "detalle": "Indicadores de coordenadas y estados sospechosos",
    "filas_antes": len(df_clean),
    "filas_despues": len(df_clean)
})

## 12.7. Resumen de indicadores de calidad

Se realiza una revisión conjunta de los indicadores generados durante la limpieza.

El objetivo es conocer la frecuencia de cada tipo de anomalía y comprobar que los indicadores se mantienen correctamente.

Se analizarán los problemas relacionados con:

- Precios anunciados.
- Superficies.
- Dormitorios y baños.
- Años de construcción.
- Coordenadas y estado.

Los indicadores pueden solaparse, por lo que sus cantidades no deben sumarse para calcular el número de viviendas afectadas.

In [ ]:
# Actualizar los indicadores afectados por las transformaciones
df_clean["superficie_invalida"] = (
    df_clean["sqft"].isna()
    | (df_clean["sqft"] <= 0)
)

df_clean["banos_sospechosos"] = (
    df_clean["baths"].isna()
    | (df_clean["baths"] < 0)
    | (df_clean["baths"] > 20)
)

# Lista de indicadores de calidad
indicadores_calidad = [
    "precio_sospechoso",
    "superficie_invalida",
    "superficie_extrema",
    "dormitorios_sospechosos",
    "dormitorios_cero",
    "banos_sospechosos",
    "banos_cero_original",
    "construccion_sospechosa",
    "construccion_muy_antigua",
    "construccion_futura",
    "coordenadas_sospechosas",
    "estado_sospechoso"
]

# Resumen de indicadores
resumen_calidad = pd.DataFrame({
    "Registros afectados": df_clean[indicadores_calidad].sum()
})

resumen_calidad["Porcentaje (%)"] = (
    resumen_calidad["Registros afectados"]
    / len(df_clean) * 100
).round(2)

display(
    resumen_calidad.sort_values(
        "Registros afectados",
        ascending=False
    )
)

# Número de viviendas con al menos una anomalía
alguna_anomalia = df_clean[indicadores_calidad].any(axis=1)

print(
    "Viviendas con al menos un indicador activo:",
    alguna_anomalia.sum()
)

print(
    "Viviendas sin indicadores activos:",
    (~alguna_anomalia).sum()
)

print("Total de viviendas:", len(df_clean))

## 12.8. Corrección y validación de indicadores originales

Durante la revisión de calidad se detectó que el indicador `banos_cero_original` no conservaba correctamente los valores iniciales.

Esto puede ocurrir cuando una celda de transformación se ejecuta varias veces y vuelve a calcular un indicador sobre datos ya modificados.

Para corregirlo, se recuperan los indicadores desde `df_single`, que conserva las características originales de las viviendas unifamiliares.

También se comprueba que los identificadores de los registros siguen alineados entre los DataFrames.

Esta corrección mejora la trazabilidad y la reproducibilidad del proceso.

In [ ]:
# Comprobar que los registros siguen alineados
assert df_clean.index.equals(df_single.index), (
    "Los índices no coinciden. Revisar antes de continuar."
)

# Recuperar los indicadores desde los datos originales
df_clean["banos_cero_original"] = (
    df_single["baths"] == 0
)

df_clean["dormitorios_cero"] = (
    df_single["beds"] == 0
)

# Recuperar también el indicador de superficies cero
df_clean["superficie_cero_original"] = (
    df_single["sqft"] == 0
)

# Validar resultados
print(
    "Baños originalmente iguales a cero:",
    df_clean["banos_cero_original"].sum()
)

print(
    "Dormitorios originalmente iguales a cero:",
    df_clean["dormitorios_cero"].sum()
)

print(
    "Superficies originalmente iguales a cero:",
    df_clean["superficie_cero_original"].sum()
)

print("Registros conservados:", len(df_clean))

## 12.9. Validaciones automáticas de limpieza

Se realizan comprobaciones automáticas para verificar que las transformaciones se han aplicado correctamente.

Las validaciones comprueban:

- Conservación de los registros originales seleccionados.
- Ausencia de superficies iguales a cero después de la normalización.
- Ausencia de baños iguales a cero después de la normalización.
- Conservación de los indicadores originales.
- Ausencia de identificadores duplicados.
- Conservación de las 422.006 viviendas unifamiliares.

Estas pruebas permiten detectar errores antes de generar la capa Silver.

In [ ]:
# Validaciones automáticas
assert len(df_clean) == filas_iniciales, (
    "Ha cambiado el número de registros."
)

assert (df_clean["sqft"] == 0).sum() == 0, (
    "Todavía existen superficies iguales a cero."
)

assert (df_clean["baths"] == 0).sum() == 0, (
    "Todavía existen baños iguales a cero."
)

assert df_clean["banos_cero_original"].sum() == 1097, (
    "El indicador original de baños no es correcto."
)

assert df_clean["superficie_cero_original"].sum() == 755, (
    "El indicador original de superficies no es correcto."
)

assert df_clean["id"].is_unique, (
    "Existen identificadores duplicados."
)

assert df_clean["property_type"].eq("single_family").all(), (
    "Existen propiedades de otros tipos."
)

print("Todas las validaciones superadas.")
print("Registros:", len(df_clean))
print("Columnas actuales:", len(df_clean.columns))

## 12.10. Comparación antes y después de la limpieza

Se comparan las principales variables del dataset original de viviendas unifamiliares (`df_single`) con el dataset transformado (`df_clean`).

El objetivo es comprobar qué valores se han modificado y demostrar que las transformaciones aplicadas son coherentes con las decisiones documentadas.

Se compararán:

- Número de registros.
- Valores ausentes.
- Valores iguales a cero.
- Valores negativos.
- Número de columnas.

La comparación permitirá verificar que los cambios realizados corresponden a las reglas de limpieza establecidas.

In [ ]:
# Variables que queremos comparar
variables_comparacion = [
    "list_price",
    "sqft",
    "beds",
    "baths",
    "year_built"
]

# Crear una lista para guardar los resultados
resultados_comparacion = []

for columna in variables_comparacion:

    resultados_comparacion.append({
        "Variable": columna,
        "Nulos antes": df_single[columna].isna().sum(),
        "Nulos después": df_clean[columna].isna().sum(),
        "Ceros antes": (df_single[columna] == 0).sum(),
        "Ceros después": (df_clean[columna] == 0).sum(),
        "Negativos antes": (df_single[columna] < 0).sum(),
        "Negativos después": (df_clean[columna] < 0).sum()
    })

# Convertir los resultados en una tabla
tabla_comparacion = pd.DataFrame(resultados_comparacion)

display(tabla_comparacion)

# Comparar dimensiones
print("DIMENSIONES ANTES:")
print(df_single.shape)

print("\nDIMENSIONES DESPUÉS:")
print(df_clean.shape)

### 12.10.1. Verificación de conservación de los datos originales

Se comprueba que las transformaciones no hayan modificado accidentalmente otras variables.

Las únicas columnas originales cuyos valores se han normalizado son:

- `sqft`: superficies iguales a cero.
- `baths`: baños iguales a cero.

También se han normalizado los tipos de datos de algunas columnas, sin que esto deba alterar su significado.

Las demás columnas originales deben conservar su información.

Los indicadores de calidad se mantienen como columnas adicionales.

In [ ]:
# Columnas originales que no deberían haber cambiado
columnas_sin_cambios = [
    "list_price",
    "beds",
    "year_built",
    "status",
    "property_type",
    "lat",
    "lng",
    "id"
]

# Comprobar igualdad de cada columna
for columna in columnas_sin_cambios:

    iguales = df_single[columna].equals(
        df_clean[columna]
    )

    print(f"{columna}: {'OK' if iguales else 'REVISAR'}")

    assert iguales, (
        f"Se detectaron cambios inesperados en {columna}"
    )

print("\nComprobación de conservación superada.")

## 12.11. Preparación del dataset Silver

La capa Silver contendrá las viviendas unifamiliares seleccionadas, las transformaciones justificadas y los indicadores de calidad generados.

Se conservarán las variables originales útiles para el análisis y los indicadores que permitan identificar registros que requieran revisión.

La capa Silver no representa todavía el conjunto definitivo de entrenamiento del modelo.

Antes de exportarla se revisarán los tipos de datos, los valores ausentes y la estructura general del DataFrame.

In [ ]:
# Mostrar los tipos de datos actuales
print("TIPOS DE DATOS DEL DATASET LIMPIO")

display(
    df_clean.dtypes.to_frame(
        name="Tipo de dato"
    )
)

# Resumir los tipos de datos
print("\nRESUMEN DE TIPOS")

display(
    df_clean.dtypes.astype(str).value_counts()
)

# Comprobar columnas de fecha
print("\nTIPO DE LA FECHA DE ÚLTIMA VENTA")

print(df_clean["last_sold_date"].dtype)

# Comprobar identificadores
print("\nIDENTIFICADORES")

print("IDs ausentes:", df_clean["id"].isna().sum())
print("IDs duplicados:", df_clean["id"].duplicated().sum())

# Dimensiones finales
print("\nDIMENSIONES")

print(df_clean.shape)

### 12.11.1. Revisión de valores ausentes en Silver

Se realiza una última revisión de los valores ausentes del dataset limpio.

Esta comprobación permitirá identificar qué columnas tienen una elevada proporción de información incompleta.

Las columnas con muchos valores ausentes no se eliminarán automáticamente, ya que algunas pueden resultar útiles para análisis posteriores.

La decisión de utilizar o descartar cada variable dependerá de su significado, calidad y utilidad para el proyecto.

In [ ]:
# Calcular los valores ausentes
nulos_silver = pd.DataFrame({
    "Nulos": df_clean.isna().sum(),
    "Porcentaje (%)": (
        df_clean.isna().mean() * 100
    ).round(2)
})

# Ordenar de mayor a menor
nulos_silver = nulos_silver.sort_values(
    "Porcentaje (%)",
    ascending=False
)

# Mostrar columnas con valores ausentes
display(
    nulos_silver[nulos_silver["Nulos"] > 0]
)

# Mostrar resumen
print(
    "Columnas con valores ausentes:",
    (df_clean.isna().sum() > 0).sum()
)

print(
    "Columnas sin valores ausentes:",
    (df_clean.isna().sum() == 0).sum()
)

print(
    "Total de columnas:",
    len(df_clean.columns)
)

### 12.11.2. Revisión de variables de características inmobiliarias

Se revisan las columnas que describen características adicionales de las viviendas y que actualmente tienen tipo `object`.

Las variables analizadas son:

- `central_air`
- `pool`
- `garage`
- `waterfront`
- `fireplace`
- `basement`

El objetivo es comprobar si contienen valores booleanos, numéricos o categóricos y decidir si necesitan una normalización adicional.

Los valores ausentes se conservarán hasta conocer su significado, evitando interpretarlos automáticamente como ausencia de la característica.

In [ ]:
# Columnas que queremos revisar
columnas_caracteristicas = [
    "central_air",
    "pool",
    "garage",
    "waterfront",
    "fireplace",
    "basement"
]

# Mostrar los valores de cada columna
for columna in columnas_caracteristicas:

    print(f"\nVARIABLE: {columna}")

    print("Tipo de dato:", df_clean[columna].dtype)

    print("Valores ausentes:", df_clean[columna].isna().sum())

    display(
        df_clean[columna].value_counts(
            dropna=False
        ).head(15)
    )

### 12.11.3. Comprobación de integridad de columnas originales

Se verifica que las columnas originales del dataset conservan sus valores después de la limpieza, excepto aquellas en las que se han realizado transformaciones justificadas.

Las modificaciones previstas son:

- `sqft`: sustitución de ceros por valores ausentes.
- `baths`: sustitución de ceros por valores ausentes.
- `last_sold_date`: conversión al tipo fecha.
- `zip`: conversión al tipo texto.
- Otras variables numéricas: normalización del tipo de dato.

Las nuevas columnas de indicadores no forman parte del dataset original.

Esta comprobación permite detectar cambios accidentales antes de generar la capa Silver.

In [ ]:
# Columnas originales que no hemos transformado
columnas_transformadas = [
    "sqft",
    "baths",
    "last_sold_date",
    "zip",
    "beds",
    "lot_sqft",
    "year_built",
    "list_price",
    "last_sold_price",
    "hoa_fee"
]

columnas_sin_transformar = [
    columna
    for columna in df_single.columns
    if columna not in columnas_transformadas
]

# Comparar todas las columnas sin transformar
columnas_modificadas = []

for columna in columnas_sin_transformar:

    if not df_single[columna].equals(df_clean[columna]):
        columnas_modificadas.append(columna)

print(
    "Columnas originales comprobadas:",
    len(columnas_sin_transformar)
)

print(
    "Columnas con cambios inesperados:",
    columnas_modificadas
)

assert len(columnas_modificadas) == 0, (
    "Existen modificaciones inesperadas."
)

print("Integridad de columnas originales verificada.")

### 12.11.4. Normalización de características booleanas

Las variables `central_air`, `pool`, `garage`, `waterfront`, `fireplace` y `basement` contienen valores `True` y valores ausentes.

Se ha comprobado que no aparecen valores `False` explícitos.

Por tanto, no es correcto interpretar los valores ausentes como ausencia de la característica.

**Decisión de limpieza:**

- Convertir las seis variables al tipo `boolean` de pandas.
- Conservar los valores `True`.
- Mantener los valores ausentes como `pd.NA`.
- No eliminar registros por la ausencia de estas características.

Esta transformación permite representar correctamente tres estados posibles: verdadero, falso y desconocido, aunque en este dataset solo se hayan observado verdadero y desconocido.

In [ ]:
# Columnas de características inmobiliarias
columnas_booleanas = [
    "central_air",
    "pool",
    "garage",
    "waterfront",
    "fireplace",
    "basement"
]

# Guardar los recuentos originales
recuentos_antes = pd.DataFrame({
    "True": [
        df_clean[col].eq(True).sum()
        for col in columnas_booleanas
    ],
    "Nulos": [
        df_clean[col].isna().sum()
        for col in columnas_booleanas
    ]
}, index=columnas_booleanas)

# Convertir al tipo booleano nullable
for columna in columnas_booleanas:
    df_clean[columna] = df_clean[columna].astype("boolean")

# Comprobar que los valores se han conservado
recuentos_despues = pd.DataFrame({
    "True": [
        df_clean[col].eq(True).sum()
        for col in columnas_booleanas
    ],
    "Nulos": [
        df_clean[col].isna().sum()
        for col in columnas_booleanas
    ]
}, index=columnas_booleanas)

assert recuentos_antes.equals(recuentos_despues), (
    "Han cambiado los valores de alguna característica."
)

print("Conversión completada correctamente.")

display(df_clean[columnas_booleanas].dtypes)

display(recuentos_despues)

print("Registros conservados:", len(df_clean))

## 12.12. Definición del conjunto candidato para Machine Learning

El objetivo provisional de HousingPulse AI es estimar el precio anunciado (`list_price`) de viviendas unifamiliares en Miami-Dade.

La capa Silver conservará los registros originales seleccionados, las transformaciones justificadas y los indicadores de calidad.

Sin embargo, para entrenar un modelo será necesario seleccionar viviendas que cumplan unos requisitos mínimos.

Antes de aplicar filtros, se analizará la distribución de los precios según el estado del anuncio (`status`).

Este análisis permitirá estudiar si determinados estados presentan precios anormalmente bajos y definir una primera propuesta de selección.

No se asumirá que un precio bajo corresponde necesariamente a un alquiler sin evidencia adicional.

In [ ]:
# Analizar los precios por estado del anuncio
resumen_precios_estado = (
    df_clean.groupby("status", dropna=False)
    .agg(
        viviendas=("id", "count"),
        precio_mediano=("list_price", "median"),
        precio_medio=("list_price", "mean"),
        precio_minimo=("list_price", "min")
    )
)

# Contar los precios sospechosos por estado
precios_sospechosos_estado = (
    df_clean.groupby("status", dropna=False)["precio_sospechoso"]
    .sum()
)

# Añadir el resultado a la tabla
resumen_precios_estado["precios_sospechosos"] = (
    precios_sospechosos_estado
)

# Calcular el porcentaje de precios sospechosos
resumen_precios_estado["porcentaje_sospechoso"] = (
    resumen_precios_estado["precios_sospechosos"]
    / resumen_precios_estado["viviendas"]
    * 100
).round(2)

display(
    resumen_precios_estado.sort_values(
        "viviendas",
        ascending=False
    )
)

### 12.12.1. Análisis de precios según el estado del anuncio

El análisis de precios por estado muestra diferencias importantes entre los registros.

El estado `other` presenta un precio mediano de 3.505 dólares y un 93,30 % de precios sospechosos.

El estado `pending` también presenta una proporción elevada de precios sospechosos (47,67 %).

Por el contrario, los estados `active`, `sold` y `off_market` presentan precios medianos considerablemente superiores.

Estos resultados sugieren que el dataset podría mezclar distintos tipos de operaciones inmobiliarias, incluyendo posibles alquileres.

Sin embargo, no se dispone todavía de información suficiente para confirmar el significado de todos los registros.

**Decisión provisional:**

- Conservar todos los registros en la capa Silver.
- No interpretar automáticamente `status = other` como alquiler.
- Comparar diferentes criterios de selección para Machine Learning.
- Documentar las viviendas excluidas por cada criterio.

### 12.12.2. Comparación de criterios para seleccionar viviendas

Se comparan diferentes criterios de selección para construir un conjunto candidato para Machine Learning.

Las alternativas consideradas son:

1. Todas las viviendas unifamiliares.
2. Viviendas con precio anunciado superior a 10.000 dólares.
3. Viviendas con precio superior a 10.000 dólares y superficie válida.
4. Viviendas con precio superior a 10.000 dólares, superficie válida y dormitorios y baños conocidos.
5. Viviendas que cumplen los requisitos anteriores y tienen un estado distinto de `other`.

El umbral de 10.000 dólares se utiliza como criterio exploratorio para identificar posibles anuncios que no correspondan a compraventas.

No se considera un límite definitivo ni una prueba de que los precios inferiores correspondan a alquileres.

El objetivo es evaluar cuántos registros conservaríamos con cada alternativa antes de tomar una decisión.

In [ ]:
# Condiciones básicas
precio_valido = (
    df_clean["list_price"].notna()
    & (df_clean["list_price"] > 10000)
)

superficie_valida = (
    df_clean["sqft"].notna()
    & (df_clean["sqft"] > 0)
)

habitaciones_validas = (
    df_clean["beds"].notna()
    & df_clean["baths"].notna()
)

estado_filtrado = (
    df_clean["status"] != "other"
)

# Definir escenarios
escenarios = {
    "1. Todas las viviendas": pd.Series(
        True,
        index=df_clean.index
    ),

    "2. Precio > 10.000": precio_valido,

    "3. Precio + superficie": (
        precio_valido
        & superficie_valida
    ),

    "4. Precio + superficie + habitaciones": (
        precio_valido
        & superficie_valida
        & habitaciones_validas
    ),

    "5. Anterior + excluir other": (
        precio_valido
        & superficie_valida
        & habitaciones_validas
        & estado_filtrado
    )
}

# Calcular resultados
resultados_escenarios = []

for nombre, condicion in escenarios.items():

    viviendas = condicion.sum()

    resultados_escenarios.append({
        "Escenario": nombre,
        "Viviendas": viviendas,
        "Porcentaje conservado": round(
            viviendas / len(df_clean) * 100,
            2
        ),
        "Viviendas excluidas": len(df_clean) - viviendas
    })

tabla_escenarios = pd.DataFrame(resultados_escenarios)

display(tabla_escenarios)

### 12.12.3. Selección provisional del conjunto candidato

Tras comparar cinco escenarios, se selecciona provisionalmente el escenario 4.

Este escenario conserva 390.464 viviendas unifamiliares, equivalentes al 92,53 % de la muestra inicial.

Los requisitos son:

- Precio anunciado superior a 10.000 dólares.
- Superficie construida conocida y positiva.
- Número de dormitorios conocido.
- Número de baños conocido.

No se excluyen automáticamente registros según el estado del anuncio.

La selección es provisional y deberá revisarse antes del entrenamiento, especialmente por la posible presencia de alquileres, precios extremos y otras inconsistencias.

El dataset Silver conservará las 422.006 viviendas y sus indicadores de calidad.

In [ ]:
# Recuperar la condición del escenario 4
condicion_ml = escenarios[
    "4. Precio + superficie + habitaciones"
]

# Crear el conjunto candidato
df_ml = df_clean.loc[condicion_ml].copy()

# Comprobar resultados
print("Viviendas originales:", len(df_single))
print("Viviendas Silver:", len(df_clean))
print("Viviendas candidatas ML:", len(df_ml))

print(
    "Porcentaje conservado:",
    round(len(df_ml) / len(df_clean) * 100, 2),
    "%"
)

# Validar los requisitos
assert (df_ml["list_price"] > 10000).all()
assert (df_ml["sqft"] > 0).all()
assert df_ml["beds"].notna().all()
assert df_ml["baths"].notna().all()

print("\nSelección provisional validada.")

### 12.12.4. Revisión del conjunto candidato para Machine Learning

Se analiza la distribución de los precios y de los estados del anuncio después de aplicar los requisitos mínimos.

El objetivo es comprobar si siguen existiendo precios extremos, anuncios de estado ambiguo u otras características que requieran una revisión adicional.

No se eliminarán registros en esta fase.

In [ ]:
# Estadísticas de precios del conjunto candidato
print("DISTRIBUCIÓN DE PRECIOS")

display(
    df_ml["list_price"].describe(
        percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    )
)

# Estados de los anuncios
print("\nESTADOS DE LOS ANUNCIOS")

display(
    df_ml["status"].value_counts(dropna=False)
)

# Valores extremos
print("\nPRECIOS EXTREMOS")

print(
    "Viviendas con precio superior a 5 millones:",
    (df_ml["list_price"] > 5_000_000).sum()
)

print(
    "Viviendas con precio superior a 10 millones:",
    (df_ml["list_price"] > 10_000_000).sum()
)

print(
    "Viviendas con superficie superior a 20.000 sqft:",
    (df_ml["sqft"] > 20_000).sum()
)

print(
    "Viviendas con anomalías geográficas:",
    (
        df_ml["coordenadas_sospechosas"]
        | df_ml["estado_sospechoso"]
    ).sum()
)

### 12.12.5. Análisis de valores extremos mediante el precio por superficie

Se analiza la relación entre el precio anunciado (`list_price`) y la superficie construida (`sqft`) del conjunto candidato para Machine Learning.

Se calcula el precio por pie cuadrado para identificar posibles incoherencias y estudiar la dispersión de los precios.

Este indicador se utiliza únicamente para el análisis de calidad.

No se eliminarán viviendas automáticamente por tener precios elevados o bajos por superficie.

Además, el precio por superficie no se utilizará como variable predictora del modelo, porque se calcula a partir de `list_price`, que es la variable que queremos predecir. Utilizarlo produciría una fuga de información.

In [ ]:
# Calcular el precio anunciado por pie cuadrado
df_ml["precio_por_sqft"] = (
    df_ml["list_price"] / df_ml["sqft"]
)

# Analizar su distribución
print("DISTRIBUCIÓN DEL PRECIO POR SQFT")

display(
    df_ml["precio_por_sqft"].describe(
        percentiles=[
            0.01, 0.05, 0.25,
            0.5, 0.75, 0.95, 0.99
        ]
    )
)

# Identificar valores extremos para revisión
print("\nVALORES EXTREMOS")

print(
    "Precio por sqft superior a 5.000 $:",
    (df_ml["precio_por_sqft"] > 5000).sum()
)

print(
    "Precio por sqft inferior a 50 $:",
    (df_ml["precio_por_sqft"] < 50).sum()
)

# Mostrar los 10 precios por sqft más elevados
print("\nVIVIENDAS CON MAYOR PRECIO POR SQFT")

display(
    df_ml.nlargest(
        10, "precio_por_sqft"
    )[
        [
            "city",
            "zip",
            "status",
            "list_price",
            "sqft",
            "precio_por_sqft",
            "beds",
            "baths"
        ]
    ]
)

### 12.12.6. Revisión de viviendas con superficies extremas

Se revisan las viviendas candidatas con superficies superiores a 20.000 pies cuadrados.

El objetivo es comprobar si estas propiedades presentan características coherentes con viviendas de gran tamaño o si existen posibles errores en el registro de la superficie.

Se conservan todos los registros durante esta revisión.

In [ ]:
# Seleccionar viviendas con superficies extremas
viviendas_grandes = df_ml.loc[
    df_ml["sqft"] > 20000
].copy()

print(
    "Viviendas con más de 20.000 sqft:",
    len(viviendas_grandes)
)

# Estadísticas de estas viviendas
display(
    viviendas_grandes[
        ["sqft", "list_price", "beds", "baths"]
    ].describe()
)

# Mostrar las 15 viviendas de mayor superficie
display(
    viviendas_grandes.nlargest(
        15, "sqft"
    )[
        [
            "city",
            "zip",
            "status",
            "sqft",
            "beds",
            "baths",
            "list_price",
            "precio_por_sqft"
        ]
    ]
)

### 12.12.7. Interpretación de valores extremos

El precio mediano por pie cuadrado del conjunto candidato es de 373,17 dólares.

Se han identificado:

- 219 viviendas con precios superiores a 5.000 dólares por sqft.
- 1.324 viviendas con precios inferiores a 50 dólares por sqft.
- 59 viviendas con superficies superiores a 20.000 sqft.

Algunos registros presentan combinaciones poco habituales de precio, superficie y número de habitaciones.

Estos resultados pueden deberse a errores de registro, diferencias en la definición de superficie o propiedades excepcionales.

**Decisión:**

1. Conservar todos los registros en Silver.
2. Identificar los valores extremos mediante indicadores.
3. Comparar un conjunto candidato general con otro que excluya provisionalmente los extremos.
4. No considerar los umbrales utilizados como límites universales de validez inmobiliaria.
5. No utilizar `precio_por_sqft` como predictor, ya que contiene información de la variable objetivo `list_price`.

### 12.12.8. Creación de indicadores de precios por superficie

Se crean indicadores para identificar viviendas con relaciones extremas entre el precio anunciado y la superficie construida.

Se consideran inicialmente dos situaciones:

- Precio por sqft inferior a 50 dólares.
- Precio por sqft superior a 5.000 dólares.

También se identifica por separado la superficie superior a 20.000 sqft.

Estos indicadores permitirán comparar diferentes criterios de selección sin modificar los datos originales.

In [ ]:
# Indicadores de precio por superficie
df_ml["precio_sqft_muy_bajo"] = (
    df_ml["precio_por_sqft"] < 50
)

df_ml["precio_sqft_muy_alto"] = (
    df_ml["precio_por_sqft"] > 5000
)

# Indicador de superficie muy elevada
df_ml["superficie_muy_grande"] = (
    df_ml["sqft"] > 20000
)

# Indicador combinado
df_ml["extremo_precio_superficie"] = (
    df_ml["precio_sqft_muy_bajo"]
    | df_ml["precio_sqft_muy_alto"]
    | df_ml["superficie_muy_grande"]
)

# Mostrar resultados
print(
    "Precio por sqft muy bajo:",
    df_ml["precio_sqft_muy_bajo"].sum()
)

print(
    "Precio por sqft muy alto:",
    df_ml["precio_sqft_muy_alto"].sum()
)

print(
    "Superficie muy grande:",
    df_ml["superficie_muy_grande"].sum()
)

print(
    "Viviendas con alguna anomalía extrema:",
    df_ml["extremo_precio_superficie"].sum()
)

print("Total de viviendas candidatas:", len(df_ml))

### 12.12.9. Comparación de conjuntos candidatos

Se comparan dos versiones del conjunto candidato para Machine Learning:

- Conjunto general: incluye las 390.464 viviendas que cumplen los requisitos básicos.
- Conjunto sin extremos: excluye provisionalmente las viviendas marcadas por los indicadores de precio por superficie y superficie muy elevada.

El objetivo es estudiar cuántos registros se perderían y cómo cambiaría la distribución de precios.

Esta comparación no determina todavía qué conjunto ofrece mejores predicciones. Para ello será necesario entrenar y evaluar modelos posteriormente.

In [ ]:
# Conjunto general
df_ml_general = df_ml.copy()

# Conjunto sin extremos
df_ml_sin_extremos = df_ml.loc[
    ~df_ml["extremo_precio_superficie"]
].copy()

# Comparar tamaños y precios
comparacion_modelos = pd.DataFrame({
    "Conjunto": [
        "General",
        "Sin extremos"
    ],
    "Viviendas": [
        len(df_ml_general),
        len(df_ml_sin_extremos)
    ],
    "Precio mediano": [
        df_ml_general["list_price"].median(),
        df_ml_sin_extremos["list_price"].median()
    ],
    "Precio medio": [
        df_ml_general["list_price"].mean(),
        df_ml_sin_extremos["list_price"].mean()
    ],
    "Precio máximo": [
        df_ml_general["list_price"].max(),
        df_ml_sin_extremos["list_price"].max()
    ]
})

display(comparacion_modelos)

print(
    "Viviendas excluidas provisionalmente:",
    len(df_ml_general) - len(df_ml_sin_extremos)
)

print(
    "Porcentaje conservado:",
    round(
        len(df_ml_sin_extremos)
        / len(df_ml_general) * 100,
        2
    ),
    "%"
)

### 12.12.10. Comparación de conjuntos con y sin valores extremos

Se han comparado dos conjuntos candidatos para Machine Learning:

- Conjunto general: 390.464 viviendas.
- Conjunto sin extremos: 388.882 viviendas.

La exclusión provisional afecta a 1.582 viviendas, equivalentes al 0,41 % del conjunto general.

El precio mediano pasa de 626.000 a 626.800 dólares, mientras que el precio medio disminuye de aproximadamente 976.754 a 960.043 dólares.

El precio máximo pasa de 237 millones a 79 millones de dólares.

**Decisión provisional:**

1. Conservar el conjunto general como referencia.
2. Conservar el conjunto sin extremos como alternativa experimental.
3. No modificar ni eliminar registros de la capa Silver.
4. Documentar los umbrales utilizados y sus limitaciones.
5. Comparar posteriormente el rendimiento de ambas alternativas mediante una evaluación adecuada.

La eliminación de extremos no se considera todavía una mejora demostrada.

### 12.12.11. Caracterización de las viviendas excluidas provisionalmente

Se analizan los registros que quedarían fuera del conjunto candidato sin extremos.

Se estudiarán:

- Su distribución por estado del anuncio.
- Sus precios anunciados.
- Sus superficies.
- Sus precios por pie cuadrado.

El objetivo es comprobar si los registros excluidos presentan características compatibles con errores de datos o con viviendas excepcionales.

Este análisis no implica la eliminación definitiva de los registros.

In [ ]:
# Seleccionar los registros excluidos provisionalmente
df_excluidos_extremos = df_ml.loc[
    df_ml["extremo_precio_superficie"]
].copy()

print(
    "Viviendas excluidas provisionalmente:",
    len(df_excluidos_extremos)
)

# Distribución por estado del anuncio
print("\nESTADOS DE LOS REGISTROS EXCLUIDOS")

display(
    df_excluidos_extremos["status"].value_counts(
        dropna=False
    )
)

# Estadísticas de las variables principales
print("\nESTADÍSTICAS DE LOS REGISTROS EXCLUIDOS")

display(
    df_excluidos_extremos[
        ["list_price", "sqft", "precio_por_sqft", "beds", "baths"]
    ].describe(
        percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]
    )
)

# Comprobar si existen anomalías geográficas
print("\nANOMALÍAS GEOGRÁFICAS")

print(
    (
        df_excluidos_extremos["coordenadas_sospechosas"]
        | df_excluidos_extremos["estado_sospechoso"]
    ).sum()
)

### 12.12.12. Conclusiones del análisis de valores extremos

El análisis ha identificado 1.582 viviendas con relaciones extremas entre precio y superficie o con superficies superiores a 20.000 sqft.

De estas viviendas, 935 presentan el estado `other`, lo que representa aproximadamente el 59,10 % de los registros señalados.

El precio mediano de las viviendas marcadas es de 27.000 dólares, mientras que el precio medio supera los 5 millones de dólares.

Esta diferencia muestra que el conjunto contiene tanto precios anormalmente bajos como propiedades con precios extraordinariamente elevados.

Los resultados sugieren que podrían existir diferentes tipos de operaciones inmobiliarias, posibles errores de registro y viviendas excepcionales.

**Decisiones:**

- Mantener las 422.006 viviendas unifamiliares en Silver.
- Conservar los indicadores de calidad.
- Mantener el conjunto general de 390.464 viviendas como referencia.
- Mantener el conjunto de 388.882 viviendas como alternativa experimental.
- No considerar demostrada la superioridad de ninguno de los conjuntos.
- Revisar la procedencia y vigencia de los precios antes de entrenar un modelo definitivo.

Los valores extremos no se eliminan de la capa Silver.

### 13. Validación final del dataset Silver

Antes de exportar el dataset Silver se realizan comprobaciones de integridad y calidad.

Se verifica que:

1. Se conservan las 422.006 viviendas unifamiliares.
2. No existen identificadores duplicados o nulos.
3. Todas las viviendas pertenecen a `single_family`.
4. Las superficies iguales a cero se han normalizado como valores ausentes.
5. Los baños iguales a cero se han normalizado como valores ausentes.
6. Las columnas booleanas utilizan un tipo que permite valores desconocidos.
7. Los indicadores de calidad se han conservado.
8. La columna `last_sold_date` tiene un tipo de fecha adecuado.

Estas validaciones permiten comprobar la coherencia técnica del conjunto Silver, aunque no garantizan que todos los datos originales sean correctos.

In [ ]:
# 1. Número de registros
assert len(df_clean) == 422006

# 2. Identificadores
assert df_clean["id"].notna().all()
assert df_clean["id"].is_unique

# 3. Tipo de propiedad
assert df_clean["property_type"].eq("single_family").all()

# 4. Normalizaciones realizadas
assert not (df_clean["sqft"] == 0).any()
assert not (df_clean["baths"] == 0).any()

# 5. Columnas booleanas
columnas_booleanas = [
    "central_air",
    "pool",
    "garage",
    "waterfront",
    "fireplace",
    "basement"
]

for columna in columnas_booleanas:
    assert str(df_clean[columna].dtype) == "boolean"

# 6. Indicadores de calidad
indicadores_obligatorios = [
    "precio_sospechoso",
    "superficie_invalida",
    "superficie_extrema",
    "dormitorios_sospechosos",
    "banos_sospechosos",
    "construccion_sospechosa",
    "coordenadas_sospechosas",
    "estado_sospechoso",
    "banos_cero_original",
    "superficie_cero_original"
]

assert all(
    columna in df_clean.columns
    for columna in indicadores_obligatorios
)

# 7. Tipo de fecha
assert pd.api.types.is_datetime64_any_dtype(
    df_clean["last_sold_date"]
)

# 8. Resumen final
print("VALIDACIÓN SILVER COMPLETADA")

print("Filas:", len(df_clean))
print("Columnas:", len(df_clean.columns))
print("IDs duplicados:", df_clean["id"].duplicated().sum())
print("IDs nulos:", df_clean["id"].isna().sum())
print("Superficies iguales a cero:", (df_clean["sqft"] == 0).sum())
print("Baños iguales a cero:", (df_clean["baths"] == 0).sum())

print("\nTodas las comprobaciones han sido superadas.")

## 14. Exportación del dataset Silver

Una vez superadas las validaciones, se prepara la exportación del conjunto Silver a formato Parquet.

Se utiliza Parquet porque permite conservar los tipos de datos y almacenar la información de manera eficiente.

El archivo Silver contiene las 422.006 viviendas unifamiliares, las transformaciones realizadas y los indicadores de calidad.

Los registros con valores sospechosos se conservan para permitir su análisis posterior.

El archivo se guardará en `data/silver/`, sin modificar el CSV original de `data/raw/`.

In [ ]:
# Comprobar si pyarrow está instalado
import importlib.util

pyarrow_disponible = (
    importlib.util.find_spec("pyarrow") is not None
)

print("PyArrow instalado:", pyarrow_disponible)

### 14.1. Preparación para la exportación a Parquet

Para exportar el dataset Silver se utiliza la biblioteca PyArrow.

Esta biblioteca permite guardar los datos en formato Parquet, conservando los tipos de datos y facilitando su posterior lectura con Pandas.

La instalación se realiza en el entorno de Python asociado al notebook.

In [ ]:
%pip install pyarrow

In [ ]:
import pyarrow as pa

print("PyArrow instalado correctamente")
print("Versión:", pa.__version__)

### 14.2. Exportación del dataset Silver

Se exporta la propuesta individual de limpieza a formato Parquet.

El dataset contiene 422.006 viviendas unifamiliares y 43 columnas, incluyendo los indicadores de calidad.

Se conservan los registros con valores sospechosos para evitar pérdidas de información y permitir que el equipo decida posteriormente los criterios definitivos.

El archivo se guarda en `data/silver/`, manteniendo intacto el CSV original.

Esta versión es provisional y está destinada a la comparación de propuestas entre los integrantes del equipo.

In [ ]:
# Crear la carpeta Silver si todavía no existe
carpeta_silver = BASE_DIR / "data" / "silver"
carpeta_silver.mkdir(parents=True, exist_ok=True)

# Definir la ruta del archivo
ruta_silver = (
    carpeta_silver
    / "housingpulse_single_family_silver_joan.parquet"
)

# Exportar el dataset
df_clean.to_parquet(
    ruta_silver,
    engine="pyarrow",
    index=False
)

# Comprobar que se ha creado
print("Archivo Silver exportado correctamente")
print("Ruta:", ruta_silver.resolve())

print(
    "Tamaño del archivo (MB):",
    round(ruta_silver.stat().st_size / (1024 ** 2), 2)
)

print("Filas exportadas:", len(df_clean))
print("Columnas exportadas:", len(df_clean.columns))

### 14.3. Verificación del archivo Silver

Se vuelve a cargar el archivo Parquet exportado para comprobar su integridad.

Se verifica que el número de filas, las columnas, los identificadores y los tipos de datos coinciden con el DataFrame original de limpieza.

Esta comprobación garantiza que la exportación no ha introducido diferencias detectables entre ambos conjuntos.

In [ ]:
# Volver a cargar el archivo exportado
df_silver_verificado = pd.read_parquet(
    ruta_silver,
    engine="pyarrow"
)

# Crear una copia para realizar la comparación
df_silver_comparacion = df_silver_verificado.copy()

# Normalizar el tipo de zip para que coincida con el original
df_silver_comparacion["zip"] = (
    df_silver_comparacion["zip"].astype(
        df_clean["zip"].dtype
    )
)

# Comparar dimensiones
assert df_silver_comparacion.shape == df_clean.shape

# Comparar columnas
assert df_silver_comparacion.columns.equals(df_clean.columns)

# Comparar tipos de datos
assert df_silver_comparacion.dtypes.equals(df_clean.dtypes)

# Comparar todo el contenido
pd.testing.assert_frame_equal(
    df_silver_comparacion,
    df_clean.reset_index(drop=True)
)

# Comprobar identificadores
assert df_silver_comparacion["id"].is_unique

print("VERIFICACIÓN DEL ARCHIVO SILVER SUPERADA")

print("Filas:", len(df_silver_comparacion))
print("Columnas:", len(df_silver_comparacion.columns))
print("IDs duplicados:", df_silver_comparacion["id"].duplicated().sum())

print("\nEl archivo exportado coincide con el DataFrame limpio.")

### 14.4. Comprobación de tipos de datos tras la exportación

La primera verificación confirma que el archivo Parquet conserva el número de filas y columnas.

Sin embargo, se ha detectado una diferencia entre los tipos de datos del DataFrame original y los del archivo recuperado.

Se comparan los tipos de datos columna por columna para identificar el origen de la diferencia antes de realizar cualquier corrección.

In [ ]:
# Comparar los tipos de datos originales y exportados
comparacion_tipos = pd.DataFrame({
    "Tipo original": df_clean.dtypes.astype(str),
    "Tipo Parquet": df_silver_verificado.dtypes.astype(str)
})

# Identificar las diferencias
comparacion_tipos["Coinciden"] = (
    comparacion_tipos["Tipo original"]
    == comparacion_tipos["Tipo Parquet"]
)

# Mostrar únicamente las columnas diferentes
diferencias_tipos = comparacion_tipos.loc[
    ~comparacion_tipos["Coinciden"]
]

print("Columnas con diferencias:", len(diferencias_tipos))

display(diferencias_tipos)

### 14.5. Normalización del tipo de código postal tras la lectura

La comparación de tipos ha identificado una única diferencia en la columna `zip`.

El DataFrame original utiliza el tipo `string`, mientras que el archivo Parquet recuperado utiliza `str`.

Se normaliza el tipo de la columna recuperada para realizar una comparación homogénea.

Esta operación no modifica el archivo Parquet exportado.

Posteriormente se comprobará que todos los valores, las columnas y los tipos de datos coinciden con el DataFrame Silver original.

In [ ]:
# Crear una copia del archivo recuperado
df_silver_comparacion = df_silver_verificado.copy()

# Normalizar el tipo del código postal
df_silver_comparacion["zip"] = (
    df_silver_comparacion["zip"].astype(
        df_clean["zip"].dtype
    )
)

# Comprobar dimensiones
assert df_silver_comparacion.shape == df_clean.shape

# Comprobar columnas
assert df_silver_comparacion.columns.equals(
    df_clean.columns
)

# Comprobar tipos
assert df_silver_comparacion.dtypes.equals(
    df_clean.dtypes
)

# Comprobar el contenido completo
pd.testing.assert_frame_equal(
    df_silver_comparacion,
    df_clean.reset_index(drop=True)
)

print("VERIFICACIÓN SILVER SUPERADA")
print("Filas:", len(df_silver_comparacion))
print("Columnas:", len(df_silver_comparacion.columns))
print("IDs duplicados:", df_silver_comparacion["id"].duplicated().sum())
print("Contenido completo: idéntico")
print("Tipos de datos: coinciden tras normalizar zip")